In [95]:
import pandas as pd
from sklearn.impute import KNNImputer
import pandas as pd
import numpy as np
import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA

In [4]:
df = pd.read_csv("final_esg_dataset.csv")
pd.set_option('display.float_format', '{:.4f}'.format)
df.head()

,Country Name,Country Code,Series Name,Series Code,2000,2001,2002,2003,2004,2005,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Afghanistan,AFG,Access to electricity (% of population),EG.ELC.ACCS.ZS,4.4000,9.3000,14.1000,19.0000,23.8000,28.7000,...,71.5000,97.7000,97.7000,93.4000,97.7000,97.7000,97.7000,85.3000,85.3000,NaN
1,Afghanistan,AFG,GDP (current US$),NY.GDP.MKTP.CD,3521418060.0000,2813571754.0000,3825701439.0000,4520946819.0000,5224896719.0000,6203256539.0000,...,19134221645.0000,18116572395.0000,18753456498.0000,18053222687.0000,18799444490.0000,19955929052.0000,14259995441.0000,14497243872.0000,17152234637.0000,17778508876.0000
2,Afghanistan,AFG,"Population, total",SP.POP.TOTL,20130327.0000,20284307.0000,21378117.0000,22733049.0000,23560654.0000,24404567.0000,...,33831764.0000,34700612.0000,35688935.0000,36743039.0000,37856121.0000,39068979.0000,40000412.0000,40578842.0000,41454761.0000,42647492.0000
3,Afghanistan,AFG,Land area (sq. km),AG.LND.TOTL.K2,652230.0000,652230.0000,652230.0000,652230.0000,652230.0000,652230.0000,...,652230.0000,652230.0000,652230.0000,652230.0000,652230.0000,652230.0000,652230.0000,652230.0000,652230.0000,652230.0000
4,Afghanistan,AFG,Voice and Accountability - Governance estimate...,GOV_WGI_VA.EST,-2.2978,NaN,-1.3641,-1.2675,-1.2310,-1.1630,...,-1.2099,-1.2165,-1.2397,-1.1858,-1.1405,-1.0815,-1.4280,-1.8416,-2.0371,-2.1484


In [5]:
df.isnull().sum()

Country Name       0
Country Code       0
Series Name        0
Series Code        0
2000            1656
2001            2567
2002            1418
2003            1426
2004            1362
2005            1152
2006            1146
2007             946
2008             916
2009             919
2010             882
2011             864
2012             858
2013             877
2014             843
2015             811
2016             829
2017             832
2018             819
2019             825
2020             861
2021            1264
2022            1899
2023            2512
2024            3248
dtype: int64

acess to electricity 

In [7]:
# Select Access to electricity
electricity = df[df["Series Code"] == "EG.ELC.ACCS.ZS"].copy()

# Get year columns
years = [col for col in df.columns if str(col).isdigit()]

# Count missing values
electricity["Missing Count"] = electricity[years].isna().sum(axis=1)

# Find missing years
electricity["Missing Years"] = electricity[years].apply(
    lambda row: ", ".join(str(year) for year in years if pd.isna(row[year])),
    axis=1
)

# Show only countries with missing values
electricity_missing = electricity[electricity["Missing Count"] > 0]

display(electricity_missing[
    ["Country Name", "Missing Count", "Missing Years"]
])

,Country Name,Missing Count,Missing Years
0,Afghanistan,1,2024
62,Albania,1,2024
92,Algeria,1,2024
132,Andorra,1,2024
161,Angola,1,2024
...,...,...,...
7022,"Venezuela, RB",1,2024
7056,Viet Nam,1,2024
7097,"Yemen, Rep.",1,2024
7126,Zambia,1,2024


In [14]:
# Load the new Excel file
new_data = pd.read_excel("acces_to_electricity.xls", header=3)

# Select Access to electricity
new_electricity = new_data[
    new_data["Indicator Code"] == "EG.ELC.ACCS.ZS"
].copy()

# Match countries using country codes
new_2024 = new_electricity.set_index("Country Code")["2024"]

# Select electricity indicator in original dataset
mask = df["Series Code"] == "EG.ELC.ACCS.ZS"

# Fill only missing 2024 values
df.loc[mask, "2024"] = df.loc[mask, "2024"].fillna(
    df.loc[mask, "Country Code"].map(new_2024)
)

# Check remaining missing values
electricity = df[df["Series Code"] == "EG.ELC.ACCS.ZS"]

print("Missing values:", electricity["2024"].isna().sum())

Missing values: 0


In [15]:
# Check missing values from 2000 to 2023
display(electricity[electricity.loc[:, "2000":"2023"].isna().any(axis=1)])

,Country Name,Country Code,Series Name,Series Code,2000,2001,2002,2003,2004,2005,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
2541,Guinea-Bissau,GNB,Access to electricity (% of population),EG.ELC.ACCS.ZS,NaN,NaN,1.3000,2.6000,3.9000,5.2000,...,20.1000,22.9000,26.0000,28.4000,30.9000,33.4000,35.8000,37.4000,40.5000,43.7000
3381,"Korea, Dem. People's Rep.",PRK,Access to electricity (% of population),EG.ELC.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,40.0000,42.1000,44.2000,46.3000,48.4000,50.5000,52.6000,54.7000,57.5000,61.4000
3702,Liberia,LBR,Access to electricity (% of population),EG.ELC.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,15.2000,17.7000,24.2000,22.8000,23.1000,27.6000,29.8000,31.8000,32.5000,34.9000
5902,South Sudan,SSD,Access to electricity (% of population),EG.ELC.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,4.8000,5.3000,4.2000,6.3000,6.8000,7.3000,7.7000,8.4000,5.4000,5.4000


In [16]:
# Select Access to electricity
electricity = df[df["Series Code"] == "EG.ELC.ACCS.ZS"].copy()

# Select year columns
years = [col for col in df.columns if str(col).isdigit()]

# Create KNN model
knn = KNNImputer(n_neighbors=5)

# Fill missing values
electricity[years] = knn.fit_transform(electricity[years])

# Check remaining missing values
print(electricity[years].isna().sum().sum())

0


In [17]:
df.loc[electricity.index, years] = electricity[years]
# Check missing electricity values
df.loc[df["Series Code"] == "EG.ELC.ACCS.ZS", years].isna().sum().sum()

np.int64(0)

gdp

In [18]:
gdp = df[df["Series Code"] == "NY.GDP.MKTP.CD"].copy()

In [19]:
# Select year columns
years = [col for col in df.columns if str(col).isdigit()]

# Count missing values for each country
gdp["Missing Count"] = gdp[years].isna().sum(axis=1)

# Show countries with missing GDP
display(
    gdp[gdp["Missing Count"] > 0][
        ["Country Name", "Country Code", "Missing Count"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count
3360,"Korea, Dem. People's Rep.",PRK,25
5881,South Sudan,SSD,17
1925,Eritrea,ERI,13
7074,"Yemen, Rep.",YEM,6
1507,Cuba,CUB,4
6300,Syrian Arab Republic,SYR,2
5391,San Marino,SMR,1


In [20]:
gdp["Missing Years"] = gdp[years].apply(
    lambda row: ", ".join(str(year) for year in years if pd.isna(row[year])),
    axis=1
)

display(
    gdp[gdp["Missing Count"] > 0][
        ["Country Name", "Missing Count", "Missing Years"]
    ]
)

,Country Name,Missing Count,Missing Years
1507,Cuba,4,"2021, 2022, 2023, 2024"
1925,Eritrea,13,"2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019..."
3360,"Korea, Dem. People's Rep.",25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007..."
5391,San Marino,1,2024
5881,South Sudan,17,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007..."
6300,Syrian Arab Republic,2,"2023, 2024"
7074,"Yemen, Rep.",6,"2019, 2020, 2021, 2022, 2023, 2024"


In [21]:
# South Sudan GDP (millions of US dollars)
south_sudan_gdp = {
    2008: 14586,
    2009: 12231,
    2010: 14465,
    2011: 17340,
    2012: 11267,
    2013: 14947,
    2014: 14608,
    2015: 15082,
    2016: 2825,
    2017: 2736,
    2018: 2559,
    2019: 4044,
    2020: 5423,
    2021: 5940,
    2022: 7533,
    2023: 7564,
    2024: 4616,
}

# Cuba GDP (US dollars)
cuba_gdp = {
    2020: 107352000000,
    2021: 126693606820,
    2022: 147113403679,
    2023: 201985998168
}

# Eritrea GDP (millions of US dollars)
eritrea_gdp = {
    2012: 2255,
    2013: 1958,
    2014: 2604,
    2015: 2016,
    2016: 2213,
    2017: 1904,
    2018: 2006,
    2019: 1982
}

# North Korea GDP (millions of US dollars)
north_korea_gdp = {
    2000: 10608,
    2001: 11022,
    2002: 10910,
    2003: 11051,
    2004: 11168,
    2005: 13031,
    2006: 13764,
    2007: 14375,
    2008: 13337,
    2009: 12035,
    2010: 13945,
    2011: 15689,
    2012: 15907,
    2013: 16565,
    2014: 17396,
    2015: 16283,
    2016: 16789,
    2017: 17365
}

# San Marino GDP (US dollars)
san_marino_gdp = {
    2024: 2_000_000_000
}

# Syria GDP (millions of US dollars)
syria_gdp = {
    2019: 22583,
    2020: 12502,
    2021: 14893,
    2022: 23738,
    2023: 19993
}

# Yemen GDP (millions of US dollars)
yemen_gdp = {
    2017: 10443,
    2018: 11494,
    2019: 12849,
    2020: 10783,
    2021: 10203,
    2022: 11107,
    2023: 9364,
    2024: 8069
}

In [22]:
# Combine the dictionaries
gdp_data = {
    "SSD": south_sudan_gdp,
    "CUB": cuba_gdp,
    "ERI": eritrea_gdp,
    "PRK": north_korea_gdp,
    "SMR": san_marino_gdp,
    "SYR": syria_gdp,
    "YEM": yemen_gdp
}

# Fill missing GDP values
for country, values in gdp_data.items():
    for year, value in values.items():

        if country not in ["CUB", "SMR"]:
            value = value * 1_000_000

        mask = (
            (df["Country Code"] == country) &
            (df["Series Code"] == "NY.GDP.MKTP.CD")
        )

        df.loc[mask, str(year)] = df.loc[mask, str(year)].fillna(value)

In [23]:
gdp = df[df["Series Code"] == "NY.GDP.MKTP.CD"]

print(gdp[years].isna().sum().sum())

22


In [24]:
# Select GDP
gdp = df[df["Series Code"] == "NY.GDP.MKTP.CD"]

# Find missing years for each country
for _, row in gdp.iterrows():
    missing = [year for year in years if pd.isna(row[year])]

    if missing:
        print(row["Country Name"], ":", missing)

Cuba : ['2024']
Eritrea : ['2020', '2021', '2022', '2023', '2024']
Korea, Dem. People's Rep. : ['2018', '2019', '2020', '2021', '2022', '2023', '2024']
South Sudan : ['2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007']
Syrian Arab Republic : ['2024']


In [25]:
# Select GDP data
gdp = df[df["Series Code"] == "NY.GDP.MKTP.CD"].copy()

# Select year columns
years = [col for col in df.columns if str(col).isdigit()]

# Apply KNN
knn = KNNImputer(n_neighbors=5)

gdp[years] = knn.fit_transform(gdp[years])

In [26]:
# Check remaining missing GDP values
print(gdp[years].isna().sum().sum())

0


In [27]:
# Update main dataset with cleaned GDP
df.loc[gdp.index, years] = gdp[years]

population

In [28]:
# Select population data
population = df[df["Series Code"] == "SP.POP.TOTL"].copy()

# Select year columns
years = [col for col in df.columns if str(col).isdigit()]

# Count missing values
population["Missing Count"] = population[years].isna().sum(axis=1)

# Display countries with missing values
display(population[population["Missing Count"] > 0][
    ["Country Name", "Country Code", "Missing Count"]
])

,Country Name,Country Code,Missing Count


land 

In [30]:
# Select land area
land = df[df["Series Code"] == "AG.LND.TOTL.K2"].copy()

# Count missing values
land["Missing Count"] = land[years].isna().sum(axis=1)

# Find missing years
land["Missing Years"] = land[years].apply(
    lambda row: ", ".join(str(year) for year in years if pd.isna(row[year])),
    axis=1
)

# Display only countries with missing values
display(
    land[land["Missing Count"] > 0][
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
6162,Sudan,SDN,12,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007..."
5884,South Sudan,SSD,12,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007..."
4375,Montenegro,MNE,6,"2000, 2001, 2002, 2003, 2004, 2005"
5534,Serbia,SRB,6,"2000, 2001, 2002, 2003, 2004, 2005"
6125,Sub-Saharan Africa (excluding high income),SSA,1,2024
6090,Sub-Saharan Africa (IDA & IBRD countries),TSS,1,2024
5847,South Asia (IDA & IBRD),TSA,1,2024
4694,Nicaragua,NIC,1,2024
1751,East Asia & Pacific (IDA & IBRD countries),TEA,1,2024
2068,Europe & Central Asia (IDA & IBRD countries),TEC,1,2024


In [31]:
# Select land area
land = df[df["Series Code"] == "AG.LND.TOTL.K2"].copy()

# Forward fill and backward fill
land[years] = land[years].ffill(axis=1).bfill(axis=1)


print(land[years].isna().sum().sum())

0


In [32]:
df.loc[land.index, years] = land[years]

voice and accountability 

In [33]:
# Select Voice and Accountability
voice = df[df["Series Code"] == "GOV_WGI_VA.EST"].copy()

# Count missing values
voice["Missing Count"] = voice[years].isna().sum(axis=1)

# Find missing years
voice["Missing Years"] = voice[years].apply(
    lambda row: ", ".join(str(year) for year in years if pd.isna(row[year])),
    axis=1
)

# Display countries with missing values
display(
    voice[voice["Missing Count"] > 0][
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
2798,IDA blend,IDB,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007..."
6124,Sub-Saharan Africa (IDA & IBRD),TSS,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007..."
2711,IBRD only,IBD,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007..."
2760,IDA & IBRD total,IBT,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007..."
2834,IDA only,IDX,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007..."
...,...,...,...,...
2469,Guatemala,GTM,1,2001
2503,Guinea,GIN,1,2001
2539,Guinea-Bissau,GNB,1,2001
2570,Guyana,GUY,1,2001


In [35]:
# Remove rows with 25 missing values
voice = voice[voice["Missing Count"] < 25].copy()

display(
    voice[voice["Missing Count"] > 0][
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ]
)

,Country Name,Country Code,Missing Count,Missing Years
4,Afghanistan,AFG,1,2001
51,Albania,ALB,1,2001
88,Algeria,DZA,1,2001
121,Andorra,AND,1,2001
159,Angola,AGO,1,2001
...,...,...,...,...
7018,"Venezuela, RB",VEN,1,2001
7052,Viet Nam,VNM,1,2001
7085,"Yemen, Rep.",YEM,1,2001
7114,Zambia,ZMB,1,2001


In [36]:
# Fill missing values using linear interpolation
voice[years] = voice[years].interpolate(
    method="linear",
    axis=1,
    limit_area="inside"
)

# Check remaining missing values
print(voice[years].isna().sum().sum())

11


In [37]:
display(voice[voice[years].isna().any(axis=1)][["Country Name"] + years])

,Country Name,2000,2001,2002,2003,2004,2005,2006,2007,2008,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
5898,South Sudan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.5732,-1.8822,-1.8670,-2.0438,-2.0356,-1.9357,-1.9422,-1.9532,-1.8555,-1.8917


In [39]:


# Apply KNN to Voice and Accountability
knn = KNNImputer(n_neighbors=5)

voice[years] = knn.fit_transform(voice[years])

print(voice[years].isna().sum().sum())

0


In [388]:
drop_rows = df[
    (df["Series Code"] == "GOV_WGI_VA.EST") &
    (~df.index.isin(voice.index))
].index

df = df.drop(drop_rows)

df.loc[voice.index, years] = voice[years]

In [389]:
check = df[df["Series Code"] == "GOV_WGI_VA.EST"]

print("Countries remaining:", len(check))
print("Missing values:", check[years].isna().sum().sum())

Countries remaining: 193
Missing values: 0


gdp growth

In [42]:
# Select GDP growth
gdp_growth = df[df["Series Code"] == "NY.GDP.MKTP.KD.ZG"].copy()

# Count missing values
gdp_growth["Missing Count"] = gdp_growth[years].isna().sum(axis=1)

# Find missing years
gdp_growth["Missing Years"] = gdp_growth[years].apply(
    lambda row: ", ".join(str(year) for year in years if pd.isna(row[year])),
    axis=1
)


pd.set_option("display.max_colwidth", None)


# Display missing data
display(
    gdp_growth[gdp_growth["Missing Count"] > 0][
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
3361,"Korea, Dem. People's Rep.",PRK,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5882,South Sudan,SSD,18,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3746,Liechtenstein,LIE,15,"2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
1646,Djibouti,DJI,14,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013"
1927,Eritrea,ERI,13,"2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
7072,"Yemen, Rep.",YEM,6,"2019, 2020, 2021, 2022, 2023, 2024"
6301,Syrian Arab Republic,SYR,2,"2023, 2024"
5,Afghanistan,AFG,1,2000
5390,San Marino,SMR,1,2024


In [46]:
growth_data = {
    "SMR": {"2024": 1.6},
    "AFG": {"2000": -5.5},

    "SYR": {"2023": -1.2, "2024": -1.5},

    "YEM": {
        "2019": 4.7, "2020": 0.3, "2021": 6.0,
        "2022": -0.5, "2023": -9.3, "2024": -1.5
    },

    "ERI": {
    "2002": 3.0,
    "2003": -2.7,
    "2004": 1.5,
    "2005": 2.6,
    "2006": -1.0,
    "2007": 1.4,
    "2008": -19.4,
    "2009": 12.2,
    "2010": 10.9,
    "2011": 25.7,
    "2012": 1.9,
    "2013": -10.5,
    "2014": 30.9,
    "2015": -20.6,
    "2016": 7.4,
    "2017": -10.0,
    "2018": 13.0,
    "2019": 3.8
    },

    "DJI": {
        "2000": 0.7, "2001": 1.8, "2002": 3.2,
        "2003": 3.2, "2004": 2.8, "2005": 3.1,
        "2006": 4.8, "2007": 5.0, "2008": 5.8,
        "2009": 1.6, "2010": 4.1, "2011": 7.3,
        "2012": 4.8, "2013": 5.0
    },

    "LIE": {
        "2010": 8.0, "2011": -4.2, "2012": -0.2,
        "2013": 5.0, "2014": 3.6, "2015": 0.3,
        "2016": 2.4, "2017": 4.5, "2018": 3.5,
        "2019": -2.2, "2020": -5.3, "2021": 18.0,
        "2022": -4.8, "2023": 3.4, "2024": 1.8
    },

    "SSD": {
        "2016": -13.3, "2017": -5.8, "2018": -2.1,
        "2019": 0.9, "2020": -6.5, "2021": 5.3,
        "2022": -5.2, "2023": 5.3, "2024": -28.1
    },

    "PRK": {
        "2000": 0.4, "2001": 3.8, "2002": 1.2,
        "2003": 1.8, "2004": 2.1, "2005": 3.8,
        "2006": -1.0, "2007": -1.2, "2008": 3.1,
        "2009": -0.9, "2010": -0.5, "2011": 0.8,
        "2012": 1.3, "2013": 1.1, "2014": 1.0,
        "2015": -1.1, "2016": 3.9, "2017": -3.5
    }
}

In [47]:
for country, values in growth_data.items():
    for year, growth in values.items():
        mask = gdp_growth["Country Code"] == country

        gdp_growth.loc[
            mask & gdp_growth[year].isna(), year
        ] = growth

In [48]:
gdp_growth["Missing Count"] = gdp_growth[years].isna().sum(axis=1)

gdp_growth["Missing Years"] = gdp_growth[years].apply(
    lambda row: ", ".join(year for year in years if pd.isna(row[year])),
    axis=1
)

pd.set_option("display.max_colwidth", None)

display(
    gdp_growth[gdp_growth["Missing Count"] > 0][
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ]
)

,Country Name,Country Code,Missing Count,Missing Years
3361,"Korea, Dem. People's Rep.",PRK,7,"2018, 2019, 2020, 2021, 2022, 2023, 2024"
5882,South Sudan,SSD,9,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008"


In [49]:
knn = KNNImputer(n_neighbors=5)

gdp_growth[years] = knn.fit_transform(gdp_growth[years])

print(gdp_growth[years].isna().sum().sum())

0


In [50]:
df.loc[gdp_growth.index, years] = gdp_growth[years]

total green house emmsion 

In [51]:
ghg = df[df["Series Code"] == "EN.GHG.ALL.PC.CE.AR5"].copy()

In [52]:
ghg["Missing Count"] = ghg[years].isna().sum(axis=1)

ghg["Missing Years"] = ghg[years].apply(
    lambda row: ", ".join(year for year in years if pd.isna(row[year])),
    axis=1
)

pd.set_option("display.max_colwidth", None)

display(
    ghg[ghg["Missing Count"] > 0][
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
120,Andorra,AND,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3760,Liechtenstein,LIE,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4321,Monaco,MCO,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4391,Montenegro,MNE,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5405,San Marino,SMR,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5545,Serbia,SRB,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5895,South Sudan,SSD,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"


In [53]:
ghg = ghg[ghg[years].isna().sum(axis=1) < 25].copy()

df = df.drop(
    df[(df["Series Code"] == "EN.GHG.ALL.PC.CE.AR5") &
       (df[years].isna().all(axis=1))].index
)



print(ghg[years].isna().sum().sum())

0


In [64]:
df.loc[ghg.index, years] = ghg[years]
print(df[df["Series Code"] == "EN.GHG.ALL.PC.CE.AR5"][years].isna().sum().sum())

0


school enrollment 

In [54]:
school = df[df["Series Code"] == "SE.PRM.ENRR"].copy()

In [56]:
# Count missing values
school["Missing Count"] = school[years].isna().sum(axis=1)

# Find missing years
school["Missing Years"] = school[years].apply(
    lambda row: ", ".join(year for year in years if pd.isna(row[year])),
    axis=1
)

# Show all missing years
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

display(
    school[school["Missing Count"] > 0][
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
1659,Djibouti,DJI,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4248,"Middle East, North Africa, Afghanistan & Pakistan (IDA & IBRD)",TMN,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2604,Haiti,HTI,24,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5784,Somalia,SOM,23,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2024"
3373,"Korea, Dem. People's Rep.",PRK,23,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2019, 2020, 2021, 2022, 2023, 2024"
5893,South Sudan,SSD,22,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2012, 2013, 2014, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023"
4319,Monaco,MCO,21,"2001, 2002, 2003, 2004, 2005, 2006, 2007, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2017, 2018, 2019, 2020, 2021, 2022, 2023"
3020,Iraq,IRQ,21,"2001, 2002, 2005, 2006, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2219,Gabon,GAB,20,"2000, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2020, 2021, 2022, 2023, 2024"
3723,Libya,LBY,19,"2000, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"


In [57]:
# Find rows with more than 20 missing years
drop_rows = school[school["Missing Count"] > 20].index

# Remove them
school = school.drop(drop_rows)
df = df.drop(drop_rows)

school[years] = school[years].interpolate(
    axis=1,
    limit_area="inside"
)


print("Missing values remaining:", school[years].isna().sum().sum())

Missing values remaining: 318


In [58]:
# Count missing values
school["Missing Count"] = school[years].isna().sum(axis=1)

# Find missing years
school["Missing Years"] = school[years].apply(
    lambda row: ", ".join(year for year in years if pd.isna(row[year])),
    axis=1
)

# Show all missing years
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

display(
    school[school["Missing Count"] > 0][
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
6664,Turkmenistan,TKM,19,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018"
3723,Libya,LBY,19,"2000, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5648,Singapore,SGP,17,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2024"
2534,Guinea-Bissau,GNB,14,"2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3198,Japan,JPN,14,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2024"
819,Brazil,BRA,12,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011"
4283,Moldova,MDA,10,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009"
7080,"Yemen, Rep.",YEM,9,"2000, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5438,Sao Tome and Principe,STP,9,"2000, 2001, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
1903,Equatorial Guinea,GNQ,9,"2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"


In [59]:
singapore = {
    "2024": 100.7,
    "2023": 100.2,
    "2022": 99.6,
    "2021": 99.9,
    "2020": 100.5,
    "2019": 100.7,
    "2018": 100.3,
    "2017": 100.6,
    "2016": 100.8,
    "2015": 100.9,
    "2014": 101.1,
    "2013": 101.3,
    "2012": 101.9,
    "2011": 101.8,
    "2010": 101.8,
    "2009": 98.0,
    "2008": 98.1,
    "2007": 98.1,
    "2006": 97.4,
    "2005": 98.1,
    "2004": 95.7,
    "2003": 95.8,
    "2002": 95.6,
    "2001": 95.4,
    "2000": 96.6
}


for year, value in singapore.items():
    mask = school["Country Code"] == "SGP"
    school.loc[mask & school[year].isna(), year] = value

In [60]:
school = school[~school["Country Code"].isin(["TKM", "LBY"])].copy()

In [61]:

knn = KNNImputer(n_neighbors=5)

school[years] = knn.fit_transform(school[years])

In [62]:
print("Missing values:", school[years].isna().sum().sum())

Missing values: 0


In [63]:
# Remove school enrollment rows that are no longer in school
drop_rows = df[
    (df["Series Code"] == "SE.PRM.ENRR") &
    (~df.index.isin(school.index))
].index

df = df.drop(drop_rows)

df.loc[school.index, years] = school[years]

Rule of Law - Governance estimate (approx. -2.5 to +2.5)

In [66]:
rule = df[df["Series Code"] == "GOV_WGI_RL.EST"].copy()

In [67]:
# Count missing values
rule["Missing Count"] = rule[years].isna().sum(axis=1)

# Find missing years
rule["Missing Years"] = rule[years].apply(
    lambda row: ", ".join(year for year in years if pd.isna(row[year])),
    axis=1
)

# Show all missing data
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

display(
    rule[rule["Missing Count"] > 0][
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
2797,IDA blend,IDB,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
6117,Sub-Saharan Africa (IDA & IBRD),TSS,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2714,IBRD only,IBD,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2756,IDA & IBRD total,IBT,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2805,IDA only,IDX,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2864,IDA total,IDA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5855,South Asia (IDA & IBRD),TSA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4245,"Middle East, North Africa, Afghanistan & Pakistan (IDA & IBRD)",TMN,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
1760,East Asia & Pacific (IDA & IBRD),TEA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3541,Latin America & Caribbean (IDA & IBRD),TLA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"


In [68]:
rule = rule[rule["Missing Count"] < 25].copy()

display(
    rule[rule["Missing Count"] > 0][
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ]
)

,Country Name,Country Code,Missing Count,Missing Years
9,Afghanistan,AFG,1,2001
46,Albania,ALB,1,2001
80,Algeria,DZA,1,2001
117,Andorra,AND,1,2001
150,Angola,AGO,1,2001
185,Antigua and Barbuda,ATG,1,2001
222,Argentina,ARG,1,2001
258,Armenia,ARM,1,2001
290,Australia,AUS,1,2001
325,Austria,AUT,1,2001


In [69]:
# Select all countries except South Sudan
mask = rule["Country Code"] != "SSD"

# Apply linear interpolation
rule.loc[mask, years] = rule.loc[mask, years].interpolate(
    method="linear",
    axis=1,
    limit_area="inside"
)

In [70]:
knn = KNNImputer(n_neighbors=5)

# Use other countries to estimate missing values
filled = knn.fit_transform(rule[years])

# Copy only South Sudan's missing values
mask = rule["Country Code"] == "SSD"

rule.loc[mask, years] = filled[mask.to_numpy()]

In [71]:
print("Missing values:", rule[years].isna().sum().sum())

Missing values: 0


In [72]:
drop_rows = df[
    (df["Series Code"] == "GOV_WGI_RL.EST") &
    (~df.index.isin(rule.index))
].index

df = df.drop(drop_rows)



df.loc[rule.index, years] = rule[years]


print(df[df["Series Code"] == "GOV_WGI_RL.EST"][years].isna().sum().sum())

0


Research and development expenditure (% of GDP)

In [73]:
rd = df[df["Series Code"] == "GB.XPD.RSDV.GD.ZS"].copy()

In [74]:
# Count missing values
rd["Missing Count"] = rd[years].isna().sum(axis=1)

# Find missing years
rd["Missing Years"] = rd[years].apply(
    lambda row: ", ".join(year for year in years if pd.isna(row[year])),
    axis=1
)

# Show all missing data
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

display(
    rd[rd["Missing Count"] > 0][
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
10,Afghanistan,AFG,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
1657,Djibouti,DJI,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3967,Maldives,MDV,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3722,Libya,LBY,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3686,Liberia,LBR,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3617,Lebanon,LBN,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3372,"Korea, Dem. People's Rep.",PRK,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3337,Kiribati,KIR,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2863,IDA total,IDA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2806,IDA only,IDX,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"


In [75]:
rd = rd[rd["Missing Count"] < 20].copy()


display(rd[rd["Missing Count"] > 0][
    ["Country Name", "Country Code", "Missing Count", "Missing Years"]
])

,Country Name,Country Code,Missing Count,Missing Years
82,Algeria,DZA,17,"2000, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2018, 2019, 2020, 2021, 2022"
221,Argentina,ARG,1,2024
292,Australia,AUS,13,"2001, 2003, 2005, 2007, 2009, 2012, 2014, 2016, 2018, 2020, 2022, 2023, 2024"
327,Austria,AUT,1,2024
572,Belgium,BEL,1,2024
712,Bolivia,BOL,18,"2003, 2004, 2005, 2006, 2007, 2008, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2018, 2020, 2022, 2023, 2024"
747,Bosnia and Herzegovina,BIH,5,"2000, 2001, 2002, 2010, 2011"
817,Brazil,BRA,1,2024
886,Bulgaria,BGR,1,2024
927,Burkina Faso,BFA,10,"2000, 2006, 2010, 2011, 2012, 2013, 2015, 2016, 2018, 2022"


In [76]:
knn = KNNImputer(n_neighbors=5)

rd[years] = knn.fit_transform(rd[years])

print("Missing values:", rd[years].isna().sum().sum())

Missing values: 0


In [77]:
drop_rows = df[
    (df["Series Code"] == "GB.XPD.RSDV.GD.ZS") &
    (~df.index.isin(rd.index))
].index

df = df.drop(drop_rows)

df.loc[rd.index, years] = rd[years]

Renewable energy consumption (% of total final energy consumption)

In [78]:

renewable = df[df["Series Code"] == "EG.FEC.RNEW.ZS"].copy()

In [88]:
# Count missing values
renewable["Missing Count"] = renewable[years].isna().sum(axis=1)

# Find missing years
renewable["Missing Years"] = renewable[years].apply(
    lambda row: ", ".join(year for year in years if pd.isna(row[year])),
    axis=1
)

# Show all missing data
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

display(
    renewable[renewable["Missing Count"] > 0][
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
5891,South Sudan,SSD,15,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2022, 2023, 2024"
4386,Montenegro,MNE,8,"2000, 2001, 2002, 2003, 2004, 2022, 2023, 2024"
3542,Latin America & Caribbean (IDA & IBRD),TLA,4,"2021, 2022, 2023, 2024"
2794,IDA blend,IDB,4,"2021, 2022, 2023, 2024"
6150,Sub-Saharan Africa (excluding high income),SSA,4,"2021, 2022, 2023, 2024"
4246,"Middle East, North Africa, Afghanistan & Pakistan (IDA & IBRD)",TMN,4,"2021, 2022, 2023, 2024"
5856,South Asia (IDA & IBRD),TSA,4,"2021, 2022, 2023, 2024"
6451,Timor-Leste,TLS,4,"2000, 2001, 2023, 2024"
2076,Europe & Central Asia (IDA & IBRD),TEC,4,"2021, 2022, 2023, 2024"
2706,IBRD only,IBD,4,"2021, 2022, 2023, 2024"


In [89]:
renewable = renewable[renewable["Missing Count"] < 25].copy()

In [87]:
display(
    renewable[renewable["Missing Count"] > 0][
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ]
)

,Country Name,Country Code,Missing Count,Missing Years
11,Afghanistan,AFG,2,"2023, 2024"
52,Albania,ALB,3,"2022, 2023, 2024"
81,Algeria,DZA,3,"2022, 2023, 2024"
122,Andorra,AND,2,"2023, 2024"
151,Angola,AGO,3,"2022, 2023, 2024"
186,Antigua and Barbuda,ATG,2,"2023, 2024"
227,Argentina,ARG,3,"2022, 2023, 2024"
256,Armenia,ARM,3,"2022, 2023, 2024"
291,Australia,AUS,3,"2022, 2023, 2024"
326,Austria,AUT,3,"2022, 2023, 2024"


In [93]:
print(renewable[years].isna().all())

2000    False
2001    False
2002    False
2003    False
2004    False
2005    False
2006    False
2007    False
2008    False
2009    False
2010    False
2011    False
2012    False
2013    False
2014    False
2015    False
2016    False
2017    False
2018    False
2019    False
2020    False
2021    False
2022    False
2023     True
2024     True
dtype: bool


In [99]:
knn = KNNImputer(n_neighbors=5)

knn_years = [str(year) for year in range(2000, 2022)]

renewable[knn_years] = knn.fit_transform(renewable[knn_years])

In [100]:

for i, row in renewable.iterrows():

    # Historical data from 2000 to 2021
    data = row[knn_years].astype(float)

    # Create ARIMA model
    model = ARIMA(data.to_numpy(), order=(1, 1, 0))
    result = model.fit()

    # Predict 2022, 2023 and 2024
    prediction = result.forecast(steps=3)

    # Fill only missing values
    for year, value in zip(["2022", "2023", "2024"], prediction):
        if pd.isna(renewable.loc[i, year]):
            renewable.loc[i, year] = max(0, min(100, value))

c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\sari

In [101]:
print(renewable[years].isna().sum())

2000    0
2001    0
2002    0
2003    0
2004    0
2005    0
2006    0
2007    0
2008    0
2009    0
2010    0
2011    0
2012    0
2013    0
2014    0
2015    0
2016    0
2017    0
2018    0
2019    0
2020    0
2021    0
2022    0
2023    0
2024    0
dtype: int64


In [102]:
display(
    renewable[["Country Name", "2021", "2022", "2023", "2024"]]
)

,Country Name,2021,2022,2023,2024
11,Afghanistan,20.0000,20.0000,20.0000,20.0000
52,Albania,41.9000,41.7993,41.7952,41.7951
81,Algeria,0.1000,0.1000,0.1000,0.1000
122,Andorra,18.4000,18.7000,18.5776,18.6275
151,Angola,52.9000,54.2965,54.0256,54.0782
186,Antigua and Barbuda,0.9000,0.9000,0.9000,0.9000
227,Argentina,9.2000,9.0880,9.0671,9.0632
256,Armenia,9.1000,9.1831,9.1930,9.1941
291,Australia,12.3000,12.6800,12.8113,12.8567
326,Austria,36.0000,36.0764,36.1056,36.1168


In [104]:
drop_rows = df[
    (df["Series Code"] == "EG.FEC.RNEW.ZS") &
    (~df.index.isin(renewable.index))
].index

df = df.drop(drop_rows)
df.loc[renewable.index, years] = renewable[years]

print(df[df["Series Code"] == "EG.FEC.RNEW.ZS"][years].isna().sum().sum())

0


Renewable electricity output (% of total electricity output)

In [107]:
electricity_output = df[df["Series Code"] == "EG.ELC.RNEW.ZS"].copy()

In [108]:
# Count missing values
electricity_output["Missing Count"] = electricity_output[years].isna().sum(axis=1)

# Find missing years
electricity_output["Missing Years"] = electricity_output[years].apply(
    lambda row: ", ".join(year for year in years if pd.isna(row[year])),
    axis=1
)

# Display all missing data
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

display(
    electricity_output[electricity_output["Missing Count"] > 0][
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
5407,San Marino,SMR,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3762,Liechtenstein,LIE,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4322,Monaco,MCO,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5057,Paraguay,PRY,19,"2004, 2006, 2007, 2008, 2009, 2010, 2011, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5897,South Sudan,SSD,15,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2022, 2023, 2024"
3517,Lao PDR,LAO,15,"2000, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2013, 2014, 2022, 2023, 2024"
45,Albania,ALB,14,"2008, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2047,Ethiopia,ETH,13,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2012, 2013, 2021, 2022, 2023, 2024"
2004,Eswatini,SWZ,13,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2015, 2022, 2023, 2024"
7122,Zambia,ZMB,12,"2001, 2002, 2003, 2006, 2007, 2008, 2009, 2010, 2011, 2022, 2023, 2024"


In [109]:
print(electricity_output[years].isna().sum())

2000     12
2001     14
2002     12
2003     12
2004     12
2005      9
2006      9
2007      9
2008     11
2009      8
2010      8
2011      9
2012      6
2013     10
2014      9
2015      7
2016      6
2017      8
2018      8
2019      8
2020      7
2021      7
2022    205
2023    205
2024    205
dtype: int64


In [110]:
electricity_output = electricity_output[
    electricity_output[years].isna().sum(axis=1) < 25
].copy()

In [111]:
# Select years 2000 to 2021
knn_years = [str(year) for year in range(2000, 2022)]

# Apply KNN
knn = KNNImputer(n_neighbors=5)

electricity_output[knn_years] = knn.fit_transform(
    electricity_output[knn_years]
)

In [112]:
for i, row in electricity_output.iterrows():

    # Historical data from 2000 to 2021
    data = row[knn_years].astype(float)

    try:
        # Train ARIMA
        model = ARIMA(data.to_numpy(), order=(1, 1, 0))
        result = model.fit()

        # Predict 2022, 2023 and 2024
        prediction = result.forecast(steps=3)

        # Fill only missing values
        for year, value in zip(["2022", "2023", "2024"], prediction):
            if pd.isna(electricity_output.loc[i, year]):
                electricity_output.loc[i, year] = max(0, min(100, value))

    except ValueError:
        print("Could not forecast:", row["Country Name"])

c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive paramete

In [113]:
print(electricity_output[years].isna().sum())

2000    0
2001    0
2002    0
2003    0
2004    0
2005    0
2006    0
2007    0
2008    0
2009    0
2010    0
2011    0
2012    0
2013    0
2014    0
2015    0
2016    0
2017    0
2018    0
2019    0
2020    0
2021    0
2022    0
2023    0
2024    0
dtype: int64


In [114]:
display(
    electricity_output[
        ["Country Name", "2021", "2022", "2023", "2024"]
    ]
)

,Country Name,2021,2022,2023,2024
12,Afghanistan,78.2345,76.6378,76.3510,76.2995
45,Albania,92.3397,92.4852,92.5097,92.5138
87,Algeria,0.9136,0.9749,0.9496,0.9601
114,Andorra,93.3404,93.7395,93.7001,93.7040
157,Angola,91.1643,92.4097,92.6458,92.6906
192,Antigua and Barbuda,5.6558,6.0541,6.2783,6.4046
219,Argentina,25.5361,25.2638,25.2208,25.2140
262,Armenia,30.0381,29.5271,29.5667,29.5636
297,Australia,26.6609,29.5155,31.5223,32.9330
332,Austria,79.9114,80.0022,79.9946,79.9953


In [115]:
drop_rows = df[
    (df["Series Code"] == "EG.ELC.RNEW.ZS") &
    (~df.index.isin(electricity_output.index))
].index

df = df.drop(drop_rows)

df.loc[electricity_output.index, years] = electricity_output[years]

Ratio of female to male labor force participation rate (%) (modeled ILO estimate)

In [116]:
labor = df[df["Series Code"] == "SL.TLF.CACT.FM.ZS"].copy()

In [117]:
# Count missing values
labor["Missing Count"] = labor[years].isna().sum(axis=1)

# Find missing years
labor["Missing Years"] = labor[years].apply(
    lambda row: ", ".join(year for year in years if pd.isna(row[year])),
    axis=1
)

# Show all missing data
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

display(
    labor[labor["Missing Count"] > 0][
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
113,Andorra,AND,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
183,Antigua and Barbuda,ATG,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
6693,Tuvalu,TUV,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5993,St. Kitts and Nevis,KNA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5568,Seychelles,SYC,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5398,San Marino,SMR,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4943,Palau,PLW,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4557,Nauru,NRU,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4308,Monaco,MCO,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4203,"Micronesia, Fed. Sts.",FSM,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"


In [118]:
labor = labor[labor[years].isna().sum(axis=1) < 25].copy()

knn = KNNImputer(n_neighbors=5)

labor[years] = knn.fit_transform(labor[years])






In [119]:
drop_rows = df[
    (df["Series Code"] == "SL.TLF.CACT.FM.ZS") &
    (~df.index.isin(labor.index))
].index

df = df.drop(drop_rows)

df.loc[labor.index, years] = labor[years]


print(df[df["Series Code"] == "SL.TLF.CACT.FM.ZS"][years].isna().sum().sum())

0


Proportion of seats held by women in national parliaments (%)

In [120]:
parliament = df[df["Series Code"] == "SG.GEN.PARL.ZS"].copy()

In [121]:
# Count missing values
parliament["Missing Count"] = parliament[years].isna().sum(axis=1)

# Find missing years
parliament["Missing Years"] = parliament[years].apply(
    lambda row: ", ".join(year for year in years if pd.isna(row[year])),
    axis=1
)

# Show all missing data
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

display(
    parliament[parliament["Missing Count"] > 0][
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
847,Brunei Darussalam,BRN,16,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015"
4486,Myanmar,MMR,13,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2022, 2023, 2024"
5887,South Sudan,SSD,11,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010"
14,Afghanistan,AFG,9,"2000, 2001, 2002, 2003, 2004, 2018, 2022, 2023, 2024"
2107,Fiji,FJI,9,"2000, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013"
2492,Guinea,GIN,8,"2002, 2008, 2009, 2010, 2011, 2012, 2013, 2021"
6167,Sudan,SDN,7,"2000, 2019, 2020, 2021, 2022, 2023, 2024"
2598,Haiti,HTI,6,"2000, 2020, 2021, 2022, 2023, 2024"
3717,Libya,LBY,6,"2000, 2001, 2002, 2003, 2004, 2011"
4383,Montenegro,MNE,6,"2000, 2001, 2002, 2003, 2004, 2005"


In [122]:
# Select countries with exactly 1 missing value
mask = parliament[years].isna().sum(axis=1) == 1

# Apply linear interpolation
parliament.loc[mask, years] = parliament.loc[mask, years].interpolate(
    axis=1,
    limit_direction="both"
)

In [123]:
knn = KNNImputer(n_neighbors=5)

parliament[years] = knn.fit_transform(parliament[years])

In [124]:
df.loc[parliament.index, years] = parliament[years]

print(df[df["Series Code"] == "SG.GEN.PARL.ZS"][years].isna().sum().sum())

0


In [ ]:
Prevalence of undernourishment (% of population)

In [125]:
undernourishment = df[df["Series Code"] == "SN.ITK.DEFC.ZS"].copy()

In [126]:
# Count missing values
undernourishment["Missing Count"] = undernourishment[years].isna().sum(axis=1)

# Find missing years
undernourishment["Missing Years"] = undernourishment[years].apply(
    lambda row: ", ".join(year for year in years if pd.isna(row[year])),
    axis=1
)

# Show all missing data
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

display(
    undernourishment[undernourishment["Missing Count"] > 0][
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
1931,Eritrea,ERI,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3191,Japan,JPN,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
955,Burundi,BDI,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5641,Singapore,SGP,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
6516,Tonga,TON,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
846,Brunei Darussalam,BRN,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4312,Monaco,MCO,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4941,Palau,PLW,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4569,Nauru,NRU,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
671,Bhutan,BTN,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"


In [127]:
undernourishment_data = {
    "JPN": {
        "2022": 3.4, "2021": 3.2, "2020": 2.9,
        "2019": 2.8, "2018": 2.6, "2017": 2.7,
        "2016": 2.7, "2015": 2.7, "2014": 2.5,
        "2013": 2.5, "2012": 2.6, "2011": 2.9,
        "2010": 3.2, "2009": 3.0, "2008": 2.5,
        "2007": 2.5, "2006": 2.5, "2005": 2.5,
        "2004": 2.5, "2003": 2.5, "2002": 2.5,
        "2001": 2.5
    },

    "YEM": {
        "2022": 39.5, "2021": 36.9, "2020": 36.7,
        "2019": 38.4, "2018": 40.2, "2017": 42.0,
        "2016": 41.3, "2015": 38.1, "2014": 32.4,
        "2013": 28.5, "2012": 28.1, "2011": 25.2,
        "2010": 24.2, "2009": 22.2, "2008": 23.7,
        "2007": 23.8, "2006": 24.4, "2005": 25.5,
        "2004": 26.0, "2003": 25.7, "2002": 25.3,
        "2001": 24.4
    },

    "SSD": {
        "2022": 19.6,
        "2021": 19.3,
        "2020": 19.6
    },

    "SDN": {
        "2022": 11.4, "2021": 11.0, "2020": 11.1,
        "2019": 10.8, "2018": 10.9, "2017": 10.5,
        "2016": 10.0, "2015": 9.2, "2014": 8.8,
        "2013": 9.2
    },

    "CUB": {
        "2022": 2.5, "2021": 2.5, "2020": 2.5,
        "2019": 2.5, "2018": 2.5, "2017": 2.5,
        "2016": 2.5, "2015": 2.5, "2014": 2.5,
        "2013": 2.5, "2012": 2.5, "2011": 2.5,
        "2010": 2.5, "2009": 2.5, "2008": 2.5,
        "2007": 2.5, "2006": 2.5, "2005": 2.5,
        "2004": 2.5, "2003": 2.5, "2002": 2.5,
        "2001": 2.5
    },

    "PRK": {
        "2018": 45.3, "2017": 45.8, "2016": 44.2,
        "2015": 43.3, "2014": 42.7, "2013": 41.8,
        "2012": 41.8, "2011": 41.6, "2010": 41.6,
        "2009": 41.3, "2008": 40.5, "2007": 39.1,
        "2006": 36.9, "2005": 34.5, "2004": 33.2,
        "2003": 32.9, "2002": 35.3, "2001": 36.0
    }
}

In [128]:
for country, values in undernourishment_data.items():
    for year, value in values.items():

        mask = undernourishment["Country Code"] == country

        undernourishment.loc[
            mask & undernourishment[year].isna(), year
        ] = value

In [129]:
undernourishment["Missing Count"] = undernourishment[years].isna().sum(axis=1)

undernourishment["Missing Years"] = undernourishment[years].apply(
    lambda row: ", ".join(year for year in years if pd.isna(row[year])),
    axis=1
)

display(
    undernourishment[undernourishment["Missing Count"] > 0][
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
1931,Eritrea,ERI,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
6027,St. Lucia,LCA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
1896,Equatorial Guinea,GNQ,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
955,Burundi,BDI,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4569,Nauru,NRU,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
6516,Tonga,TON,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
846,Brunei Darussalam,BRN,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4312,Monaco,MCO,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
671,Bhutan,BTN,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
6691,Tuvalu,TUV,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"


In [130]:
undernourishment = undernourishment[
    undernourishment[years].isna().sum(axis=1) < 20
].copy()

In [131]:
print(undernourishment[years].isna().all())

2000     True
2001    False
2002    False
2003    False
2004    False
2005    False
2006    False
2007    False
2008    False
2009    False
2010    False
2011    False
2012    False
2013    False
2014    False
2015    False
2016    False
2017    False
2018    False
2019    False
2020    False
2021    False
2022    False
2023    False
2024     True
dtype: bool


In [133]:

# Countries to use KNN
countries = ["CUB", "SDN", "PRK", "NIC"]

# Years 2001 to 2023
knn_years = [str(year) for year in range(2001, 2024)]

# Apply KNN using all countries as potential neighbours
knn = KNNImputer(n_neighbors=5)

filled = knn.fit_transform(undernourishment[knn_years])

# Only update the four selected countries
mask = undernourishment["Country Code"].isin(countries)

undernourishment.loc[mask, knn_years] = filled[mask.to_numpy()]

In [134]:

for i, row in undernourishment.iterrows():

    data = row[years].astype(float)

    if data.isna().sum() > 0:

        try:
            model = ARIMA(data, order=(1, 0, 0))
            result = model.fit()

            prediction = result.predict(start=0, end=len(years)-1)

            # Fill missing values only
            undernourishment.loc[i, years] = data.fillna(prediction).values

        except Exception:
            print("Could not estimate:", row["Country Name"])

c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-

In [135]:
print(undernourishment[years].isna().sum())

2000    0
2001    0
2002    0
2003    0
2004    0
2005    0
2006    0
2007    0
2008    0
2009    0
2010    0
2011    0
2012    0
2013    0
2014    0
2015    0
2016    0
2017    0
2018    0
2019    0
2020    0
2021    0
2022    0
2023    0
2024    0
dtype: int64


In [136]:
drop_rows = df[
    (df["Series Code"] == "SN.ITK.DEFC.ZS") &
    (~df.index.isin(undernourishment.index))
].index

df = df.drop(drop_rows)

df.loc[undernourishment.index, years] = undernourishment[years]

Political Stability - Governance estimate (approx. -2.5 to +2.5)

In [137]:
political = df[df["Series Code"] == "GOV_WGI_PV.EST"].copy()

In [138]:
# Count missing values
political["Missing Count"] = political[years].isna().sum(axis=1)

# Find missing years
political["Missing Years"] = political[years].apply(
    lambda row: ", ".join(year for year in years if pd.isna(row[year])),
    axis=1
)

# Show all missing data
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

display(
    political[political["Missing Count"] > 0][
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
2813,IDA only,IDX,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4240,"Middle East, North Africa, Afghanistan & Pakistan (IDA & IBRD)",TMN,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
1755,East Asia & Pacific (IDA & IBRD),TEA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5850,South Asia (IDA & IBRD),TSA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2700,IBRD only,IBD,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2749,IDA & IBRD total,IBT,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2788,IDA blend,IDB,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
6110,Sub-Saharan Africa (IDA & IBRD),TSS,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
6156,Sub-Saharan Africa (excluding high income),SSA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2856,IDA total,IDA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"


In [139]:
political = political[political[years].isna().sum(axis=1) < 25].copy()

In [140]:
political[years] = political[years].interpolate(
    method="linear",
    axis=1,
    limit_area="inside"
)

In [142]:
display(
    political[political[years].isna().any(axis=1)][
        ["Country Name", "Country Code"] + years
    ]
)

,Country Name,Country Code,2000,2001,2002,2003,2004,2005,2006,2007,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
3331,Kiribati,KIR,NaN,NaN,NaN,0.9514,0.8244,1.3245,1.3280,1.3337,...,0.9559,0.9514,0.9520,0.9559,1.0993,1.1750,1.1647,1.2429,1.2433,1.2382
4065,Marshall Islands,MHL,NaN,NaN,NaN,0.9514,0.9475,0.9706,0.9803,0.9964,...,1.1425,0.8824,1.1385,1.1424,1.1509,1.0395,1.0046,1.2820,1.2824,1.2772
4206,"Micronesia, Fed. Sts.",FSM,NaN,NaN,NaN,0.4721,0.7455,1.0803,1.0828,1.0869,...,1.0719,1.0686,1.0690,1.2059,1.0094,1.1207,1.2269,1.5042,1.5046,1.4995
4311,Monaco,MCO,NaN,NaN,NaN,0.9514,0.9183,0.9293,0.9340,0.9416,...,1.2059,1.2026,1.5452,1.5480,1.5489,1.2062,1.2044,1.3786,1.3791,1.2387
4555,Nauru,NRU,NaN,NaN,NaN,0.9514,0.9475,0.9706,0.9803,0.9964,...,1.0147,1.0102,0.8242,0.8080,0.7596,0.7591,0.7572,0.9276,1.0357,1.1047
4940,Palau,PLW,NaN,NaN,NaN,0.9514,0.9475,0.9706,0.9803,0.9964,...,0.9559,0.9514,1.1385,1.1424,1.1436,1.1430,1.1404,1.2446,1.2451,1.2387
5395,San Marino,SMR,NaN,NaN,NaN,0.9514,0.9475,0.9706,0.9803,0.9964,...,1.3291,1.3245,1.3251,1.3290,1.2067,1.2062,1.2044,1.3786,1.3791,1.3727
5885,South Sudan,SSD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-2.2793,-2.3334,-2.4456,-2.3307,-2.3911,-2.1055,-2.2195,-1.6426,-1.5297,-1.4896
5990,St. Kitts and Nevis,KNA,NaN,NaN,NaN,0.9514,1.1734,1.2433,1.2458,0.7854,...,0.6639,0.6609,0.6612,0.6639,0.6647,0.9165,0.9148,0.9177,0.9182,0.9121
6515,Tonga,TON,NaN,NaN,NaN,0.4721,0.5993,0.5996,0.4798,0.3331,...,0.9379,0.9346,0.8010,0.8562,1.0103,1.1157,1.1119,1.3675,1.2305,1.2548


In [144]:
knn = KNNImputer(n_neighbors=5)

political[years] = knn.fit_transform(political[years])

print("Missing values:", political[years].isna().sum().sum())

Missing values: 0


In [145]:
drop_rows = df[
    (df["Series Code"] == "GOV_WGI_PV.EST") &
    (~df.index.isin(political.index))
].index

df = df.drop(drop_rows)

In [146]:
df.loc[political.index, years] = political[years]

PM2.5 air pollution, mean annual exposure (micrograms per cubic meter)

In [148]:
pm25 = df[df["Series Code"] == "EN.ATM.PM25.MC.M3"].copy()

In [149]:
# Count missing values
pm25["Missing Count"] = pm25[years].isna().sum(axis=1)

# Find missing years
pm25["Missing Years"] = pm25[years].apply(
    lambda row: ", ".join(year for year in years if pd.isna(row[year])),
    axis=1
)

# Show all missing data
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

display(
    pm25[pm25["Missing Count"] > 0][
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
3748,Liechtenstein,LIE,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
17,Afghanistan,AFG,4,"2021, 2022, 2023, 2024"
4938,Palau,PLW,4,"2021, 2022, 2023, 2024"
4588,Nepal,NPL,4,"2021, 2022, 2023, 2024"
4623,Netherlands,NLD,4,"2021, 2022, 2023, 2024"
4660,New Zealand,NZL,4,"2021, 2022, 2023, 2024"
4706,Nicaragua,NIC,4,"2021, 2022, 2023, 2024"
4741,Niger,NER,4,"2021, 2022, 2023, 2024"
4763,Nigeria,NGA,4,"2021, 2022, 2023, 2024"
4811,North Macedonia,MKD,4,"2021, 2022, 2023, 2024"


In [155]:
pm25 = pm25[pm25[years].isna().sum(axis=1) < 25].copy()

In [156]:
new_pm25 = pd.read_excel("API_EN.ATM.PM25.MC.M3_DS2_en_excel_v2_650.xls", header=3)

In [151]:
# Select the PM2.5 indicator from the new Excel file
new_pm25 = new_pm25[
    new_pm25["Indicator Code"] == "EN.ATM.PM25.MC.M3"
]

# Match countries using their codes
new_pm25 = new_pm25.set_index("Country Code")

# Fill missing values year by year
for year in years:
    if year in new_pm25.columns:
        values = pm25["Country Code"].map(new_pm25[year])
        pm25[year] = pm25[year].fillna(values)

In [159]:
for i, row in pm25.iterrows():

    # Historical data from 2000 to 2023
    data = row[[str(year) for year in range(2000, 2024)]].astype(float)

    # Only predict if 2024 is missing
    if pd.isna(row["2024"]) and data.notna().sum() >= 8:

        try:
            model = ARIMA(data, order=(1, 1, 0))
            result = model.fit()

            prediction = result.forecast(steps=1).iloc[0]

            # Fill missing 2024 value
            pm25.loc[i, "2024"] = max(0, prediction)

        except Exception:
            print("Could not predict:", row["Country Name"])

c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-

In [160]:
print(pm25[years].isna().sum())

2000    0
2001    0
2002    0
2003    0
2004    0
2005    0
2006    0
2007    0
2008    0
2009    0
2010    0
2011    0
2012    0
2013    0
2014    0
2015    0
2016    0
2017    0
2018    0
2019    0
2020    0
2021    0
2022    0
2023    0
2024    0
dtype: int64


In [161]:
drop_rows = df[
    (df["Series Code"] == "EN.ATM.PM25.MC.M3") &
    (~df.index.isin(pm25.index))
].index

df = df.drop(drop_rows)


df.loc[pm25.index, years] = pm25[years]

People using safely managed sanitation services (% of population)

In [162]:
sanitation = df[df["Series Code"] == "SH.STA.SMSS.ZS"].copy()

In [163]:
# Count missing values
sanitation["Missing Count"] = sanitation[years].isna().sum(axis=1)

# Find missing years
sanitation["Missing Years"] = sanitation[years].apply(
    lambda row: ", ".join(year for year in years if pd.isna(row[year])),
    axis=1
)

# Show all missing data
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

display(
    sanitation[sanitation["Missing Count"] > 0][
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
156,Angola,AGO,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3376,"Korea, Dem. People's Rep.",PRK,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
191,Antigua and Barbuda,ATG,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4076,Marshall Islands,MHL,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4113,Mauritania,MRT,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4146,Mauritius,MUS,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4210,"Micronesia, Fed. Sts.",FSM,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4286,Moldova,MDA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4533,Namibia,NAM,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4708,Nicaragua,NIC,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"


In [165]:
sanitation_total = {


        "KHM": {
        "2001": 5.77891
    },

    "RUS": {
        "2000": 55.41942,
        "2001": 55.68408
    }, 
    
    "SAU": {
        "2000": 79.15316,
        "2001": 79.18927,
        "2002": 79.22507,
        "2003": 79.26071,
        "2004": 79.29604,
        "2005": 79.33154,
        "2006": 79.36704,
        "2007": 79.40223,
        "2008": 79.43727,
        "2009": 79.47169,
        "2010": 79.50595,
        "2011": 79.54037,
        "2012": 79.57479,
        "2013": 79.60954,
        "2014": 79.64429,
        "2015": 79.67904,
        "2016": 79.71397,
        "2017": 79.74889,
        "2018": 79.78398,
        "2019": 79.81891,
        "2020": 79.85400,
        "2021": 79.88895,
        "2022": 79.92390
    },

    "HND": {
        "2000": 43.53374,
        "2001": 44.17157,
        "2002": 44.78168,
        "2003": 45.37678,
        "2004": 45.95778,
        "2005": 46.45063,
        "2006": 46.92751,
        "2007": 47.38880,
        "2008": 47.83445,
        "2009": 48.26509,
        "2010": 48.68033,
        "2011": 49.08084,
        "2012": 49.46619,
        "2013": 49.83731,
        "2014": 50.19374,
        "2015": 50.53676,
        "2016": 50.86615,
        "2017": 51.18235,
        "2018": 51.48584,
        "2019": 51.77686,
        "2020": 52.05568,
        "2021": 52.32280,
        "2022": 52.60606
    },

    "COD": {
        "2000": 22.01330,
        "2001": 21.55332,
        "2002": 21.09717,
        "2003": 20.64489,
        "2004": 20.19653,
        "2005": 19.75201,
        "2006": 19.31554,
        "2007": 18.88752,
        "2008": 18.46801,
        "2009": 17.99044,
        "2010": 17.51831,
        "2011": 17.05157,
        "2012": 16.59013,
        "2013": 16.13391,
        "2014": 15.68284,
        "2015": 15.23683,
        "2016": 14.79576,
        "2017": 14.35951,
        "2018": 13.92799,
        "2019": 13.50106,
        "2020": 13.07858,
        "2021": 13.05841,
        "2022": 13.03577
    },

    "DOM": {
        "2000": 52.48764,
        "2001": 52.73251,
        "2002": 52.28934,
        "2003": 51.80904,
        "2004": 51.31630,
        "2005": 50.82812,
        "2006": 50.34395,
        "2007": 49.86385,
        "2008": 49.38761,
        "2009": 48.91561,
        "2010": 48.44709,
        "2011": 47.98948,
        "2012": 47.53933,
        "2013": 47.09123,
        "2014": 46.64486,
        "2015": 46.20000,
        "2016": 45.75655,
        "2017": 45.31452,
        "2018": 44.87464,
        "2019": 44.43659,
        "2020": 44.00008,
        "2021": 43.56470,
        "2022": 43.13019
    }





}

In [176]:
for country, values in sanitation_total.items():
    for year, value in values.items():

        mask = (
            (sanitation["Country Code"] == country) &
            (sanitation[year].isna())
        )

        sanitation.loc[mask, year] = value

In [177]:
sanitation["Missing Count"] = sanitation[years].isna().sum(axis=1)

sanitation["Missing Years"] = sanitation[years].apply(
    lambda row: ", ".join(year for year in years if pd.isna(row[year])),
    axis=1
)

display(
    sanitation[sanitation["Missing Count"] > 0][
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ]
)

,Country Name,Country Code,Missing Count,Missing Years
156,Angola,AGO,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
191,Antigua and Barbuda,ATG,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
228,Argentina,ARG,8,"2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
401,"Bahamas, The",BHS,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
506,Barbados,BRB,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
611,Belize,BLZ,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
716,Bolivia,BOL,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
745,Bosnia and Herzegovina,BIH,6,"2019, 2020, 2021, 2022, 2023, 2024"
787,Botswana,BWA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
856,Brunei Darussalam,BRN,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"


In [178]:
sanitation = sanitation[
    sanitation[years].isna().sum(axis=1) < 25
].copy()

In [179]:
imputer = KNNImputer(n_neighbors=5)

sanitation[years] = imputer.fit_transform(sanitation[years])

In [180]:
# Remove sanitation rows that were dropped
drop_rows = df[
    (df["Series Code"] == "SH.STA.SMSS.ZS") &
    (~df.index.isin(sanitation.index))
].index

df = df.drop(drop_rows)

# Update the remaining sanitation values
df.loc[sanitation.index, years] = sanitation[years]

Regulatory Quality - Governance estimate (approx. -2.5 to +2.5)

In [181]:
regulatory = df[df["Series Code"] == "GOV_WGI_RQ.EST"].copy()

In [182]:
regulatory["Missing Count"] = regulatory[years].isna().sum(axis=1)

regulatory["Missing Years"] = regulatory[years].apply(
    lambda row: ", ".join(year for year in years if pd.isna(row[year])),
    axis=1
)

display(
    regulatory[
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
6152,Sub-Saharan Africa (excluding high income),SSA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3540,Latin America & Caribbean (IDA & IBRD),TLA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4244,"Middle East, North Africa, Afghanistan & Pakistan (IDA & IBRD)",TMN,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
6108,Sub-Saharan Africa (IDA & IBRD),TSS,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
1759,East Asia & Pacific (IDA & IBRD),TEA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2860,IDA total,IDA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2809,IDA only,IDX,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2704,IBRD only,IBD,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5854,South Asia (IDA & IBRD),TSA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2792,IDA blend,IDB,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"


In [183]:
regulatory = regulatory[
    regulatory[years].isna().sum(axis=1) < 25
].copy()

In [184]:
mask = regulatory[years].isna().sum(axis=1) == 1

regulatory.loc[mask, years] = (
    regulatory.loc[mask, years]
    .astype(float)
    .interpolate(axis=1, limit_area="inside")
)


imputer = KNNImputer(n_neighbors=5)

regulatory[years] = imputer.fit_transform(
    regulatory[years].astype(float)
)

In [185]:
print("Remaining missing values:", regulatory[years].isna().sum().sum())
print("Countries remaining:", len(regulatory))

Remaining missing values: 0
Countries remaining: 193


In [186]:
# Remove countries with 25 missing values
drop_rows = df[
    (df["Series Code"] == "GOV_WGI_RQ.EST") &
    (~df.index.isin(regulatory.index))
].index

df = df.drop(drop_rows)

# Update the remaining countries with cleaned values
df.loc[regulatory.index, years] = regulatory[years]

Literacy rate, adult total (% of people ages 15 and above)

In [187]:
literacy = df[df["Series Code"] == "SE.ADT.LITR.ZS"].copy()

In [188]:
literacy["Missing Count"] = literacy[years].isna().sum(axis=1)

literacy["Missing Years"] = literacy[years].apply(
    lambda row: ", ".join(year for year in years if pd.isna(row[year])),
    axis=1
)

display(
    literacy[
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
4641,Netherlands,NLD,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5697,Slovak Republic,SVK,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5172,Poland,POL,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4857,Norway,NOR,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4682,New Zealand,NZL,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4332,Monaco,MCO,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4262,"Middle East, North Africa, Afghanistan & Pakistan (IDA & IBRD)",TMN,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4227,"Micronesia, Fed. Sts.",FSM,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
1602,Czechia,CZE,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
1631,Denmark,DNK,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"


In [189]:
literacy = literacy[
    literacy[years].isna().sum(axis=1) < 20
].copy()

In [190]:


for index in literacy.index:

    data = literacy.loc[index, years].astype(float)

    if data.isna().sum() > 0:

        model = ARIMA(data, order=(1, 1, 0))
        result = model.fit()

        predicted = result.get_prediction(
            start=0,
            end=len(years) - 1
        ).predicted_mean

        predicted = pd.Series(predicted.to_numpy(), index=years)

        # Fill missing values only
        literacy.loc[index, years] = data.fillna(
            predicted.clip(0, 100)
        ).to_numpy()

c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for ARMA and trend. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.p

In [191]:
literacy["Missing Count"] = literacy[years].isna().sum(axis=1)

print("Countries remaining:", len(literacy))
print("Total missing values:", literacy[years].isna().sum().sum())

display(
    literacy[
        ["Country Name", "Country Code", "Missing Count"]
    ].sort_values("Missing Count", ascending=False)
)

Countries remaining: 77
Total missing values: 90


,Country Name,Country Code,Missing Count
5480,Saudi Arabia,SAU,19
5341,Rwanda,RWA,19
1357,"Congo, Dem. Rep.",COD,18
4745,Niger,NER,18
4472,Mozambique,MOZ,16
63,Albania,ALB,0
5522,Senegal,SEN,0
5242,Qatar,QAT,0
5137,Philippines,PHL,0
5096,Peru,PER,0


In [192]:
imputer = KNNImputer(n_neighbors=5)

literacy[years] = imputer.fit_transform(literacy[years].astype(float))

In [193]:
print("Remaining missing values:", literacy[years].isna().sum().sum())
print("Countries remaining:", len(literacy))

Remaining missing values: 0
Countries remaining: 77


In [391]:
drop_rows = df[
    (df["Series Code"] == "SE.ADT.LITR.ZS") &
    (~df.index.isin(literacy.index))
].index

df = df.drop(drop_rows)

df.loc[literacy.index, years] = literacy[years]

In [392]:
check = df[df["Series Code"] == "SE.ADT.LITR.ZS"]

print("Countries remaining:", len(check))
print("Missing values:", check[years].isna().sum().sum())

Countries remaining: 77
Missing values: 0


People using safely managed drinking water services (% of population)

In [194]:
water = df[df["Series Code"] == "SH.H2O.SMDW.ZS"].copy()

In [195]:
water["Missing Count"] = water[years].isna().sum(axis=1)

water["Missing Years"] = water[years].apply(
    lambda row: ", ".join(year for year in years if pd.isna(row[year])),
    axis=1
)

display(
    water[
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
3728,Libya,LBY,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3694,Liberia,LBR,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4989,Panama,PAN,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2609,Haiti,HTI,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4536,Namibia,NAM,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
6459,Timor-Leste,TLS,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
1244,China,CHN,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
6423,Thailand,THA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
1314,Comoros,COM,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2504,Guinea,GIN,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"


In [197]:
water = water[
    water[years].isna().sum(axis=1) < 25
].copy()

In [198]:


imputer = KNNImputer(n_neighbors=5)

water[years] = imputer.fit_transform(water[years].astype(float))

In [199]:
# Remove drinking water rows that were dropped
drop_rows = df[
    (df["Series Code"] == "SH.H2O.SMDW.ZS") &
    (~df.index.isin(water.index))
].index

df = df.drop(drop_rows)

# Update the remaining rows
df.loc[water.index, years] = water[years]

In [200]:
check = df[df["Series Code"] == "SH.H2O.SMDW.ZS"]

print("Countries remaining:", len(check))
print("Missing values:", check[years].isna().sum().sum())

Countries remaining: 148
Missing values: 0


Agricultural land (% of land area)

In [201]:
agriculture = df[df["Series Code"] == "AG.LND.AGRI.ZS"].copy()

In [202]:
agriculture["Missing Count"] = agriculture[years].isna().sum(axis=1)

agriculture["Missing Years"] = agriculture[years].apply(
    lambda row: ", ".join(year for year in years if pd.isna(row[year])),
    axis=1
)

display(
    agriculture[
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
4328,Monaco,MCO,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
6183,Sudan,SDN,13,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2024"
5904,South Sudan,SSD,13,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2024"
5554,Serbia,SRB,7,"2000, 2001, 2002, 2003, 2004, 2005, 2024"
4398,Montenegro,MNE,7,"2000, 2001, 2002, 2003, 2004, 2005, 2024"
2745,IDA & IBRD total,IBT,2,"2023, 2024"
4258,"Middle East, North Africa, Afghanistan & Pakistan (IDA & IBRD)",TMN,2,"2023, 2024"
6094,Sub-Saharan Africa (IDA & IBRD),TSS,2,"2023, 2024"
6133,Sub-Saharan Africa (excluding high income),SSA,2,"2023, 2024"
2772,IDA blend,IDB,2,"2023, 2024"


In [208]:
agriculture = agriculture[
    agriculture[years].isna().sum(axis=1) < 25
].copy()

In [209]:
excel = pd.read_excel("agriclture.xls", header=3)

In [210]:
# Make sure the year columns are strings
excel.columns = excel.columns.astype(str)

# Match countries using Country Code
excel = excel.set_index("Country Code")

# Fill only missing values
for index in agriculture.index:
    country = agriculture.loc[index, "Country Code"]

    if country in excel.index:
        for year in years:
            if year in excel.columns:
                if pd.isna(agriculture.loc[index, year]):
                    agriculture.loc[index, year] = excel.loc[country, year]

In [211]:
print("Remaining missing values:", agriculture[years].isna().sum().sum())

Remaining missing values: 49


In [212]:
agriculture[years] = (
    agriculture[years]
    .astype(float)
    .ffill(axis=1)
    .bfill(axis=1)
)

In [214]:
# Remove agricultural land rows that were dropped
drop_rows = df[
    (df["Series Code"] == "AG.LND.AGRI.ZS") &
    (~df.index.isin(agriculture.index))
].index

df = df.drop(drop_rows)

# Add cleaned agricultural land values
df.loc[agriculture.index, years] = agriculture[years]

Carbon dioxide (CO2) emissions (total) excluding LULUCF (Mt CO2e)

In [215]:
co2 = df[df["Series Code"] == "EN.GHG.CO2.MT.CE.AR5"].copy()

In [216]:
co2["Missing Count"] = co2[years].isna().sum(axis=1)

co2["Missing Years"] = co2[years].apply(
    lambda row: ", ".join(year for year in years if pd.isna(row[year])),
    axis=1
)

display(
    co2[
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
5414,San Marino,SMR,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
128,Andorra,AND,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3772,Liechtenstein,LIE,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5557,Serbia,SRB,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4399,Montenegro,MNE,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4329,Monaco,MCO,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5907,South Sudan,SSD,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4924,Pakistan,PAK,0,
4679,New Zealand,NZL,0,
4709,Nicaragua,NIC,0,


In [219]:
co2 = co2[
    co2[years].isna().sum(axis=1) < 25
].copy()

In [220]:
# Remove CO2 rows that were dropped
drop_rows = df[
    (df["Series Code"] == "EN.GHG.CO2.MT.CE.AR5") &
    (~df.index.isin(co2.index))
].index

df = df.drop(drop_rows)

# Update remaining CO2 values
df.loc[co2.index, years] = co2[years]

Control of Corruption - Governance estimate (approx. -2.5 to +2.5)

In [221]:
corruption = df[df["Series Code"] == "GOV_WGI_CC.EST"].copy()

In [222]:
corruption["Missing Count"] = corruption[years].isna().sum(axis=1)

corruption["Missing Years"] = corruption[years].apply(
    lambda row: ", ".join(
        year for year in years if pd.isna(row[year])
    ),
    axis=1
)

display(
    corruption[
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
6135,Sub-Saharan Africa (excluding high income),SSA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4260,"Middle East, North Africa, Afghanistan & Pakistan (IDA & IBRD)",TMN,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
1775,East Asia & Pacific (IDA & IBRD),TEA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2720,IBRD only,IBD,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2742,IDA & IBRD total,IBT,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5870,South Asia (IDA & IBRD),TSA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2774,IDA blend,IDB,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2827,IDA only,IDX,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
6103,Sub-Saharan Africa (IDA & IBRD),TSS,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2843,IDA total,IDA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"


In [223]:
corruption = corruption[
    corruption[years].isna().sum(axis=1) < 25
].copy()

In [224]:
corruption[years] = (
    corruption[years]
    .astype(float)
    .interpolate(axis=1, limit_area="inside")
)

In [225]:
imputer = KNNImputer(n_neighbors=5)

corruption[years] = imputer.fit_transform(
    corruption[years].astype(float)
)

In [226]:
# Remove corruption rows that were dropped
drop_rows = df[
    (df["Series Code"] == "GOV_WGI_CC.EST") &
    (~df.index.isin(corruption.index))
].index

df = df.drop(drop_rows)

# Update the remaining corruption values
df.loc[corruption.index, years] = corruption[years]



check = df[df["Series Code"] == "GOV_WGI_CC.EST"]

print("Countries remaining:", len(check))
print("Missing values:", check[years].isna().sum().sum())

Countries remaining: 193
Missing values: 0


Economic and Social Rights Performance Score

In [227]:
rights = df[df["Series Code"] == "SD.ESR.PERF.XQ"].copy()

In [228]:
rights["Missing Count"] = rights[years].isna().sum(axis=1)

rights["Missing Years"] = rights[years].apply(
    lambda row: ", ".join(
        year for year in years if pd.isna(row[year])
    ),
    axis=1
)

display(
    rights[
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
4331,Monaco,MCO,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3386,"Korea, Dem. People's Rep.",PRK,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2775,IDA blend,IDB,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2826,IDA only,IDX,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2844,IDA total,IDA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5871,South Asia (IDA & IBRD),TSA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
6102,Sub-Saharan Africa (IDA & IBRD),TSS,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
6136,Sub-Saharan Africa (excluding high income),SSA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2721,IBRD only,IBD,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
1531,Cuba,CUB,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"


In [229]:
rights = rights[
    rights[years].isna().sum(axis=1) < 25
].copy()

In [230]:
rights[years] = (
    rights[years]
    .astype(float)
    .interpolate(axis=1, limit_area="inside")
)

In [231]:


for index in rights.index:

    data = rights.loc[index, years].astype(float)

    observed = data.dropna()

    if len(observed) < 6:
        continue

    try:
        model = ARIMA(observed, order=(1, 1, 0))
        result = model.fit()

        # Forecast missing years at the end
        last_year = years.index(observed.index[-1])
        end_missing = len(years) - last_year - 1

        if end_missing > 0:
            forecast = result.forecast(steps=end_missing)

            missing_years = years[last_year + 1:]

            rights.loc[index, missing_years] = forecast.to_numpy()

        # Backcast missing years at the beginning
        first_year = years.index(observed.index[0])

        if first_year > 0:
            reversed_data = observed.iloc[::-1]

            reverse_model = ARIMA(reversed_data, order=(1, 1, 0))
            reverse_result = reverse_model.fit()

            backcast = reverse_result.forecast(steps=first_year)

            rights.loc[index, years[:first_year]] = backcast.to_numpy()[::-1]

    except (ValueError, TypeError, IndexError):
        continue

c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency -1YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it is not monotonic and so

In [232]:
imputer = KNNImputer(n_neighbors=5)

rights[years] = imputer.fit_transform(
    rights[years].astype(float)
)

In [233]:
print("Countries remaining:", len(rights))
print("Missing values:", rights[years].isna().sum().sum())

Countries remaining: 183
Missing values: 0


In [234]:
# Remove rows with 25 missing values
drop_rows = df[
    (df["Series Code"] == "SD.ESR.PERF.XQ") &
    (~df.index.isin(rights.index))
].index

df = df.drop(drop_rows)

# Add the cleaned values
df.loc[rights.index, years] = rights[years]

Energy intensity level of primary energy (MJ/$2021 PPP GDP)

In [235]:
energy_intensity = df[
    df["Series Code"] == "EG.EGY.PRIM.PP.KD"
].copy()

In [236]:
energy_intensity["Missing Count"] = energy_intensity[years].isna().sum(axis=1)

energy_intensity["Missing Years"] = energy_intensity[years].apply(
    lambda row: ", ".join(
        year for year in years if pd.isna(row[year])
    ),
    axis=1
)

display(
    energy_intensity[
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
3779,Liechtenstein,LIE,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5424,San Marino,SMR,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
6334,Syrian Arab Republic,SYR,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4339,Monaco,MCO,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5914,South Sudan,SSD,15,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2022, 2023, 2024"
4409,Montenegro,MNE,7,"2000, 2001, 2002, 2003, 2004, 2023, 2024"
4576,Nauru,NRU,6,"2000, 2001, 2002, 2003, 2023, 2024"
6474,Timor-Leste,TLS,4,"2000, 2001, 2023, 2024"
4507,Myanmar,MMR,3,"2022, 2023, 2024"
4479,Mozambique,MOZ,3,"2022, 2023, 2024"


In [240]:
excel = pd.read_excel("c0001077_20261108-235504.xlsx", na_values=[".."])

In [241]:
excel.columns = excel.columns.astype(str)

for year in years:
    if year in excel.columns:
        excel[year] = pd.to_numeric(excel[year], errors="coerce")

In [243]:
country_codes = (
    df[["Country Name", "Country Code"]]
    .drop_duplicates()
    .set_index("Country Name")["Country Code"]
    .to_dict()
)

excel["Country Code"] = excel["Country"].map(country_codes)

# Check countries that did not match
display(
    excel[excel["Country Code"].isna()][["Country"]].drop_duplicates()
)

,Country
26,Kyrgyzstan
39,Republic of Moldova
44,Slovakia


In [244]:
country_fixes = {
    "Kyrgyzstan": "KGZ",
    "Republic of Moldova": "MDA",
    "Slovakia": "SVK"
}

excel["Country Code"] = excel["Country Code"].fillna(
    excel["Country"].map(country_fixes)
)

In [245]:
# Make sure year columns are strings
excel.columns = excel.columns.astype(str)

# Remove rows without country codes
excel = excel.dropna(subset=["Country Code"]).copy()

# Set country code as the index
excel = excel.set_index("Country Code")

# Fill only missing values
for index in energy_intensity.index:

    country = energy_intensity.loc[index, "Country Code"]

    if country in excel.index:

        for year in years:

            if year in excel.columns:

                if pd.isna(energy_intensity.loc[index, year]):

                    value = excel.loc[country, year]

                    if pd.notna(value):
                        energy_intensity.loc[index, year] = value

In [246]:
energy_intensity["Missing Count"] = energy_intensity[years].isna().sum(axis=1)

energy_intensity["Missing Years"] = energy_intensity[years].apply(
    lambda row: ", ".join(
        year for year in years if pd.isna(row[year])
    ),
    axis=1
)

display(
    energy_intensity[
        energy_intensity["Missing Count"] > 0
    ][
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
6334,Syrian Arab Republic,SYR,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4339,Monaco,MCO,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5424,San Marino,SMR,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5914,South Sudan,SSD,15,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2022, 2023, 2024"
4576,Nauru,NRU,6,"2000, 2001, 2002, 2003, 2023, 2024"
4409,Montenegro,MNE,6,"2000, 2001, 2002, 2003, 2004, 2024"
6474,Timor-Leste,TLS,4,"2000, 2001, 2023, 2024"
26,Afghanistan,AFG,3,"2022, 2023, 2024"
4164,Mauritius,MUS,3,"2022, 2023, 2024"
3639,Lebanon,LBN,3,"2022, 2023, 2024"


In [247]:
energy_intensity = energy_intensity[
    energy_intensity[years].isna().sum(axis=1) < 25
].copy()

In [248]:
imputer = KNNImputer(n_neighbors=5)

energy_intensity[years] = imputer.fit_transform(
    energy_intensity[years].astype(float)
)

In [250]:
# Remove countries with 25 missing values
drop_rows = df[
    (df["Series Code"] == "EG.EGY.PRIM.PP.KD") &
    (~df.index.isin(energy_intensity.index))
].index

df = df.drop(drop_rows)

# Add the cleaned energy intensity values
df.loc[energy_intensity.index, years] = energy_intensity[years]

Food production index (2014-2016 = 100)

In [251]:
food = df[df["Series Code"] == "AG.PRD.FOOD.XD"].copy()

In [252]:
food["Missing Count"] = food[years].isna().sum(axis=1)

food["Missing Years"] = food[years].apply(
    lambda row: ", ".join(
        year for year in years if pd.isna(row[year])
    ),
    axis=1
)

display(
    food[
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
5418,San Marino,SMR,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4963,Palau,PLW,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3773,Liechtenstein,LIE,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
124,Andorra,AND,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4333,Monaco,MCO,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
6179,Sudan,SDN,14,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2023, 2024"
5908,South Sudan,SSD,14,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2023, 2024"
5558,Serbia,SRB,8,"2000, 2001, 2002, 2003, 2004, 2005, 2023, 2024"
4403,Montenegro,MNE,8,"2000, 2001, 2002, 2003, 2004, 2005, 2023, 2024"
5873,South Asia (IDA & IBRD),TSA,4,"2021, 2022, 2023, 2024"


In [264]:
excel = pd.read_excel("API_AG.PRD.FOOD.XD_DS2_en_excel_v2_2603.xls",header=3)


# Convert column names to strings
excel.columns = excel.columns.astype(str)

# Remove any spaces from column names
excel.columns = excel.columns.str.strip()

# Keep only the food production indicator
if "Indicator Code" in excel.columns:
    excel = excel[
        excel["Indicator Code"] == "AG.PRD.FOOD.XD"
    ].copy()

# Make year columns numeric
for year in years:
    if year in excel.columns:
        excel[year] = pd.to_numeric(excel[year], errors="coerce")

In [265]:
# Match countries by World Bank country code
excel = excel.set_index("Country Code")

for index in food.index:

    country = food.loc[index, "Country Code"]

    if country in excel.index:

        for year in years:

            if year in excel.columns:

                if pd.isna(food.loc[index, year]):

                    value = excel.loc[country, year]

                    if pd.notna(value):
                        food.loc[index, year] = value

In [ ]:
food["Missing Count"] = food[years].isna().sum(axis=1)

food["Missing Years"] = food[years].apply(
    lambda row: ", ".join(
        year for year in years if pd.isna(row[year])
    ),
    axis=1
)

display(
    food[
        food["Missing Count"] > 0
    ][
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

In [280]:
fao = pd.read_csv("wdi_food-production_all-countries-and-regions_horizontal-years.csv")



print(fao.columns.tolist())
display(fao.head())

['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019']


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019
0,Arab World,ARB,Food production index (2004-2006 = 100),AG.PRD.FOOD.XD,NaN,22.9183,25.3026,26.1543,26.3509,27.4915,...,110.8494,115.6294,111.1086,113.5463,111.7540,NaN,NaN,NaN,NaN,NaN
1,Caribbean small states,CSS,Food production index (2004-2006 = 100),AG.PRD.FOOD.XD,NaN,68.8382,68.8512,70.5313,73.6123,77.5858,...,101.6045,103.4908,106.6666,109.4682,109.1075,NaN,NaN,NaN,NaN,NaN
2,Central Europe and the Baltics,CEB,Food production index (2004-2006 = 100),AG.PRD.FOOD.XD,NaN,NaN,NaN,NaN,NaN,NaN,...,97.2115,102.2076,98.4340,105.9298,113.6837,NaN,NaN,NaN,NaN,NaN
3,Early-demographic dividend,EAR,Food production index (2004-2006 = 100),AG.PRD.FOOD.XD,NaN,28.6164,29.3656,30.2451,30.9312,30.6672,...,118.9485,124.0332,126.8090,131.1440,133.3714,NaN,NaN,NaN,NaN,NaN
4,East Asia & Pacific,EAS,Food production index (2004-2006 = 100),AG.PRD.FOOD.XD,NaN,18.9117,20.0542,20.8330,22.0677,23.0256,...,117.9317,121.4835,126.1247,127.8982,129.6551,NaN,NaN,NaN,NaN,NaN


In [285]:
fao.columns = fao.columns.astype(str).str.strip()
fao["Country Code"] = fao["Country Code"].str.strip()

fao = fao[
    fao["Indicator Code"] == "AG.PRD.FOOD.XD"
].copy()

In [286]:
# Convert year columns to numeric
for year in years:
    if year in fao.columns:
        fao[year] = pd.to_numeric(fao[year], errors="coerce")

# Match countries using country codes
fao = fao.set_index("Country Code")

fao_data = fao.reindex(
    index=food["Country Code"],
    columns=years
)

fao_data.index = food.index

# Fill missing values only
food[years] = food[years].combine_first(fao_data)

In [287]:
food["Missing Count"] = food[years].isna().sum(axis=1)

food["Missing Years"] = food[years].apply(
    lambda row: ", ".join(
        year for year in years if pd.isna(row[year])
    ),
    axis=1
)

display(
    food[["Country Name", "Country Code", "Missing Count", "Missing Years"]]
    .sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
5418,San Marino,SMR,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
124,Andorra,AND,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4963,Palau,PLW,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4333,Monaco,MCO,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5908,South Sudan,SSD,12,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011"
3773,Liechtenstein,LIE,8,"2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5558,Serbia,SRB,6,"2000, 2001, 2002, 2003, 2004, 2005"
4403,Montenegro,MNE,6,"2000, 2001, 2002, 2003, 2004, 2005"
1778,East Asia & Pacific (IDA & IBRD),TEA,4,"2021, 2022, 2023, 2024"
5873,South Asia (IDA & IBRD),TSA,4,"2021, 2022, 2023, 2024"


In [288]:
food = food[
    food[years].isna().sum(axis=1) < 25
].copy()

print("Countries remaining:", len(food))

Countries remaining: 201


In [289]:
imputer = KNNImputer(n_neighbors=5)

food[years] = imputer.fit_transform(
    food[years].astype(float)
)

print("Missing values remaining:", food[years].isna().sum().sum())

Missing values remaining: 0


In [290]:
drop_rows = df[
    (df["Series Code"] == "AG.PRD.FOOD.XD") &
    (~df.index.isin(food.index))
].index

df = df.drop(drop_rows)

df.loc[food.index, years] = food[years]

In [291]:
check = df[df["Series Code"] == "AG.PRD.FOOD.XD"]

print("Countries remaining:", len(check))
print("Missing values:", check[years].isna().sum().sum())

Countries remaining: 201
Missing values: 0


Adjusted savings: natural resources depletion (% of GNI)

In [292]:
resources = df[df["Series Code"] == "NY.ADJ.DRES.GN.ZS"].copy()

In [293]:
resources["Missing Count"] = resources[years].isna().sum(axis=1)

resources["Missing Years"] = resources[years].apply(
    lambda row: ", ".join(
        year for year in years if pd.isna(row[year])
    ),
    axis=1
)

display(
    resources[
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
4957,Palau,PLW,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5412,San Marino,SMR,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3768,Liechtenstein,LIE,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
126,Andorra,AND,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4047,Malta,MLT,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
198,Antigua and Barbuda,ATG,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4083,Marshall Islands,MHL,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3382,"Korea, Dem. People's Rep.",PRK,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4327,Monaco,MCO,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
6008,St. Kitts and Nevis,KNA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"


In [294]:
resources = resources[
    resources[years].isna().sum(axis=1) < 20
].copy()

print("Countries remaining:", len(resources))

Countries remaining: 191


In [298]:
for index in resources.index:

    train_years = [str(y) for y in range(2000, 2022)]

    data = resources.loc[index, train_years].astype(float)

    # Only forecast where 2022–2024 have missing values
    if resources.loc[index, ["2022", "2023", "2024"]].isna().any():

        data = data.dropna()

        if len(data) >= 5:

            try:
                model = ARIMA(data, order=(1, 1, 0))
                result = model.fit()

                last_year = int(data.index[-1])

                predictions = result.forecast(steps=2024 - last_year)

                for year in ["2022", "2023", "2024"]:

                    if pd.isna(resources.loc[index, year]):

                        step = int(year) - last_year - 1

                        resources.loc[index, year] = max(
                            NAN, predictions.iloc[step]
                        )

            except Exception as e:
                print(resources.loc[index, "Country Name"], e)

In [299]:
resources["Missing Count"] = resources[years].isna().sum(axis=1)

resources["Missing Years"] = resources[years].apply(
    lambda row: ", ".join(
        year for year in years if pd.isna(row[year])
    ),
    axis=1
)

display(
    resources[
        ["Country Name", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

print("Total missing values:", resources[years].isna().sum().sum())

,Country Name,Missing Count,Missing Years
6463,Timor-Leste,14,"2000, 2001, 2002, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014"
5794,Somalia,13,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012"
6182,Sudan,11,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010"
2877,Iceland,11,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010"
1947,Eritrea,10,"2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021"
28,Afghanistan,9,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008"
6672,Turkmenistan,8,"2012, 2013, 2014, 2015, 2016, 2017, 2020, 2021"
7023,"Venezuela, RB",7,"2015, 2016, 2017, 2018, 2019, 2020, 2021"
5553,Serbia,6,"2000, 2001, 2002, 2003, 2004, 2005"
4397,Montenegro,6,"2000, 2001, 2002, 2003, 2004, 2005"


Total missing values: 129


In [300]:
resources[years] = resources[years].astype(float).interpolate(
    axis=1,
    limit_area="inside"
)

print("Missing after interpolation:",
      resources[years].isna().sum().sum())

Missing after interpolation: 74


In [301]:


imputer = KNNImputer(n_neighbors=5)

resources[years] = imputer.fit_transform(
    resources[years].astype(float)
)

print("Missing after KNN:",
      resources[years].isna().sum().sum())

Missing after KNN: 0


In [302]:
drop_rows = df[
    (df["Series Code"] == "NY.ADJ.DRES.GN.ZS") &
    (~df.index.isin(resources.index))
].index

df = df.drop(drop_rows)

df.loc[resources.index, years] = resources[years]

Government Effectiveness - Governance estimate (approx. -2.5 to +2.5)

In [303]:
effectiveness = df[
    df["Series Code"] == "GOV_WGI_GE.EST"
].copy()

In [304]:
effectiveness["Missing Count"] = effectiveness[years].isna().sum(axis=1)

effectiveness["Missing Years"] = effectiveness[years].apply(
    lambda row: ", ".join(
        year for year in years if pd.isna(row[year])
    ),
    axis=1
)

display(
    effectiveness[
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
2848,IDA total,IDA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3561,Latin America & Caribbean (IDA & IBRD),TLA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
1780,East Asia & Pacific (IDA & IBRD),TEA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4265,"Middle East, North Africa, Afghanistan & Pakistan (IDA & IBRD)",TMN,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
6139,Sub-Saharan Africa (excluding high income),SSA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
6099,Sub-Saharan Africa (IDA & IBRD),TSS,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2822,IDA only,IDX,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2725,IBRD only,IBD,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5875,South Asia (IDA & IBRD),TSA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2779,IDA blend,IDB,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"


In [305]:
effectiveness = effectiveness[
    effectiveness[years].isna().sum(axis=1) < 25
].copy()

print("Countries remaining:", len(effectiveness))

Countries remaining: 193


In [306]:
effectiveness[years] = effectiveness[years].astype(float).interpolate(
    axis=1,
    limit_area="inside"
)

print("Missing after interpolation:",
      effectiveness[years].isna().sum().sum())

Missing after interpolation: 107


In [307]:
from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors=5)

effectiveness[years] = imputer.fit_transform(
    effectiveness[years].astype(float)
)

print("Missing after KNN:",
      effectiveness[years].isna().sum().sum())

Missing after KNN: 0


In [308]:
drop_rows = df[
    (df["Series Code"] == "GOV_WGI_GE.EST") &
    (~df.index.isin(effectiveness.index))
].index

df = df.drop(drop_rows)

df.loc[effectiveness.index, years] = effectiveness[years]

In [309]:
check = df[df["Series Code"] == "GOV_WGI_GE.EST"]

print("Countries remaining:", len(check))
print("Missing values:", check[years].isna().sum().sum())

Countries remaining: 193
Missing values: 0


Government expenditure on education, total (% of government expenditure)

In [310]:
education = df[
    df["Series Code"] == "SE.XPD.TOTL.GB.ZS"
].copy()

In [311]:
education["Missing Count"] = education[years].isna().sum(axis=1)

education["Missing Years"] = education[years].apply(
    lambda row: ", ".join(
        year for year in years if pd.isna(row[year])
    ),
    axis=1
)

display(
    education[
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
1920,Equatorial Guinea,GNQ,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
1291,Colombia,COL,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3741,Libya,LBY,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3776,Liechtenstein,LIE,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3426,"Korea, Rep.",KOR,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3216,Japan,JPN,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4266,"Middle East, North Africa, Afghanistan & Pakistan (IDA & IBRD)",TMN,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4406,Montenegro,MNE,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3040,Iraq,IRQ,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
1256,China,CHN,24,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2024"


In [312]:
education = education[
    education[years].isna().sum(axis=1) < 20
].copy()

print("Countries remaining:", len(education))

Countries remaining: 183


In [313]:
education[years] = education[years].astype(float).interpolate(
    axis=1,
    limit_area="inside"
)

print("Missing after interpolation:",
      education[years].isna().sum().sum())

Missing after interpolation: 610


In [316]:


imputer = KNNImputer(n_neighbors=5)

education[years] = imputer.fit_transform(
    education[years].astype(float)
)

print("Missing after KNN:",
      education[years].isna().sum().sum())

Missing after KNN: 0


In [317]:
drop_rows = df[
    (df["Series Code"] == "SE.XPD.TOTL.GB.ZS") &
    (~df.index.isin(education.index))
].index

df = df.drop(drop_rows)

df.loc[education.index, years] = education[years]

Labor force participation rate, total (% of total population ages 15-64) (modeled ILO estimate)

In [318]:
labour = df[
    df["Series Code"] == "SL.TLF.ACTI.ZS"
].copy()

In [319]:
labour["Missing Count"] = labour[years].isna().sum(axis=1)

labour["Missing Years"] = labour[years].apply(
    lambda row: ", ".join(
        year for year in years if pd.isna(row[year])
    ),
    axis=1
)

display(
    labour[
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
3777,Liechtenstein,LIE,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5597,Seychelles,SYC,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2447,Grenada,GRD,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4232,"Micronesia, Fed. Sts.",FSM,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4967,Palau,PLW,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
6717,Tuvalu,TUV,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4092,Marshall Islands,MHL,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
1711,Dominica,DMA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3357,Kiribati,KIR,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4582,Nauru,NRU,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"


In [320]:
labour = labour[
    labour[years].isna().sum(axis=1) < 25
].copy()

print("Countries remaining:", len(labour))

Countries remaining: 190


In [321]:
imputer = KNNImputer(n_neighbors=5)

labour[years] = imputer.fit_transform(
    labour[years].astype(float)
)

print("Missing after KNN:", labour[years].isna().sum().sum())

Missing after KNN: 0


In [322]:
drop_rows = df[
    (df["Series Code"] == "SL.TLF.ACTI.ZS") &
    (~df.index.isin(labour.index))
].index

df = df.drop(drop_rows)

df.loc[labour.index, years] = labour[years]

Level of water stress: freshwater withdrawal as a proportion of available freshwater resources

In [323]:
water_stress = df[
    df["Series Code"] == "ER.H2O.FWST.ZS"
].copy()

In [324]:
water_stress["Missing Count"] = water_stress[years].isna().sum(axis=1)

water_stress["Missing Years"] = water_stress[years].apply(
    lambda row: ", ".join(
        year for year in years if pd.isna(row[year])
    ),
    axis=1
)

display(
    water_stress[
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
4093,Marshall Islands,MHL,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
6543,Tonga,TON,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3358,Kiribati,KIR,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
6096,Sub-Saharan Africa (IDA & IBRD),TSS,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5878,South Asia (IDA & IBRD),TSA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5773,Solomon Islands,SLB,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
1783,East Asia & Pacific (IDA & IBRD),TEA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4968,Palau,PLW,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5598,Seychelles,SYC,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2098,Europe & Central Asia (IDA & IBRD),TEC,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"


In [325]:
fao = pd.read_csv("FAO,DF_SDG_6_4_2,1.0+A............csv")

fao = fao[
    (fao["ACTIVITY"] == "_T") &
    (fao["SERIES"] == "ER_H2O_STRESS") &
    (fao["UNIT_MEASURE"] == "PT")
].copy()

C:\Users\chiam\AppData\Local\Temp\ipykernel_23652\3354639774.py:1: DtypeWarning: Columns (42) have mixed types. Specify dtype option on import or set low_memory=False.
  fao = pd.read_csv("FAO,DF_SDG_6_4_2,1.0+A............csv")


In [326]:
water_stress_data = {}

for _, row in fao.iterrows():

    country = row["REF_AREA_ISO3"]
    year = row["TIME_PERIOD"]
    value = row["OBS_VALUE"]

    if pd.notna(country) and pd.notna(value):

        if country not in water_stress_data:
            water_stress_data[country] = {}

        water_stress_data[country][int(year)] = float(value)

In [327]:
before = water_stress[years].isna().sum().sum()

for country, values in water_stress_data.items():

    mask = water_stress["Country Code"] == country

    for year, value in values.items():

        year = str(year)

        if year in years:
            water_stress.loc[
                mask & water_stress[year].isna(), year
            ] = value

after = water_stress[years].isna().sum().sum()

print("Missing before:", before)
print("Missing after:", after)
print("Values filled:", before - after)

Missing before: 1217
Missing after: 1005
Values filled: 212


In [329]:
water_stress["Missing Count"] = water_stress[years].isna().sum(axis=1)

water_stress["Missing Years"] = water_stress[years].apply(
    lambda row: ", ".join(
        year for year in years if pd.isna(row[year])
    ),
    axis=1
)

display(
    water_stress[
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
6142,Sub-Saharan Africa (excluding high income),SSA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4408,Montenegro,MNE,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5878,South Asia (IDA & IBRD),TSA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5773,Solomon Islands,SLB,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4583,Nauru,NRU,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3358,Kiribati,KIR,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
1783,East Asia & Pacific (IDA & IBRD),TEA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4968,Palau,PLW,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5598,Seychelles,SYC,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
6543,Tonga,TON,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"


In [330]:
water_stress = water_stress[
    water_stress[years].isna().sum(axis=1) < 25
].copy()

print("Countries remaining:", len(water_stress))

Countries remaining: 177


In [332]:
for index in water_stress.index:

    data = water_stress.loc[index, years].astype(float)

    # Find missing years
    missing = data.isna()

    if missing.any():

        # Need at least 5 observed values
        if data.notna().sum() >= 5:

            try:
                model = ARIMA(data, order=(1, 1, 0))
                result = model.fit()

                # Predict all years, 2000–2024
                predictions = result.get_prediction(
                    start=0,
                    end=len(years)-1
                ).predicted_mean

                # Fill missing values only
                for year in years:

                    if pd.isna(water_stress.loc[index, year]):

                        prediction = predictions.loc[year]

                        if pd.notna(prediction) and prediction >= 0:
                            water_stress.loc[index, year] = prediction

            except Exception as e:
                print(water_stress.loc[index, "Country Name"], e)

c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-

Afghanistan The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Albania The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Algeria The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Andorra The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Angola The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Antigua and Barbuda The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Argentina The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().


c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-

Armenia The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Australia The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Austria The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Azerbaijan The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Bahrain The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Bangladesh The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Barbados The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Belarus The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().


c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-

Belgium The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Belize The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Benin The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Bhutan The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Bolivia The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Bosnia and Herzegovina The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Botswana The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().


... (33 outputs left out)

Sweden The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Switzerland The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Syrian Arab Republic The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Tajikistan The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Tanzania The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Thailand The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Timor-Leste The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Togo The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Trinidad and Tobago The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().


c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferre

Tunisia The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Turkiye The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Turkmenistan The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Uganda The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Ukraine The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
United Arab Emirates The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
United Kingdom The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
United States The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Uruguay The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Uzbekistan The truth value of a Series is a

c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-

In [333]:
water_stress["Missing Count"] = water_stress[years].isna().sum(axis=1)

water_stress["Missing Years"] = water_stress[years].apply(
    lambda row: ", ".join(
        year for year in years if pd.isna(row[year])
    ),
    axis=1
)

display(
    water_stress[
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

print("Total missing values:",
      water_stress[years].isna().sum().sum())

,Country Name,Country Code,Missing Count,Missing Years
6193,Sudan,SDN,13,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2024"
5913,South Sudan,SSD,13,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2024"
5003,Panama,PAN,11,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2024"
5108,Peru,PER,9,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2024"
698,Bhutan,BTN,9,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2024"
488,Bangladesh,BGD,9,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2024"
6053,St. Lucia,LCA,8,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2024"
137,Andorra,AND,8,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2024"
6438,Thailand,THA,8,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2024"
5143,Philippines,PHL,7,"2000, 2001, 2002, 2003, 2004, 2005, 2024"


Total missing values: 305


In [ ]:

train_years = [str(y) for y in range(2000, 2024)]

for index in water_stress.index:

    # Only countries missing 2024
    if pd.isna(water_stress.loc[index, "2024"]):

        data = water_stress.loc[index, train_years].astype(float)

        # Need at least 5 observations
        if data.notna().sum() >= 5:

            try:
                model = ARIMA(data, order=(1, 1, 0))
                result = model.fit()

                prediction = result.forecast(steps=1).iloc[0]

                # Negative predictions remain NaN
                if pd.notna(prediction) and prediction >= 0:
                    water_stress.loc[index, "2024"] = prediction

            except Exception as e:
                print(water_stress.loc[index, "Country Name"], e)

c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-

In [337]:
imputer = KNNImputer(n_neighbors=5)

water_stress[years] = imputer.fit_transform(
    water_stress[years].astype(float)
)

print("Missing after KNN:",
      water_stress[years].isna().sum().sum())

Missing after KNN: 0


In [338]:
drop_rows = df[
    (df["Series Code"] == "ER.H2O.FWST.ZS") &
    (~df.index.isin(water_stress.index))
].index

df = df.drop(drop_rows)

df.loc[water_stress.index, years] = water_stress[years]

Life expectancy at birth, total (years)

In [339]:
life_expectancy = df[
    df["Series Code"] == "SP.DYN.LE00.IN"
].copy()

In [340]:
life_expectancy["Missing Count"] = life_expectancy[years].isna().sum(axis=1)

life_expectancy["Missing Years"] = life_expectancy[years].apply(
    lambda row: ", ".join(
        year for year in years if pd.isna(row[year])
    ),
    axis=1
)

display(
    life_expectancy[
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
33,Afghanistan,AFG,0,
4955,Palau,PLW,0,
4605,Nepal,NPL,0,
4640,Netherlands,NLD,0,
4675,New Zealand,NZL,0,
4723,Nicaragua,NIC,0,
4758,Niger,NER,0,
4780,Nigeria,NGA,0,
4828,North Macedonia,MKD,0,
4863,Norway,NOR,0,


Gini index

In [341]:
gini = df[
    df["Series Code"] == "SI.POV.GINI"
].copy()

In [342]:
gini["Missing Count"] = gini[years].isna().sum(axis=1)

gini["Missing Years"] = gini[years].apply(
    lambda row: ", ".join(
        year for year in years if pd.isna(row[year])
    ),
    axis=1
)

display(
    gini[
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
34,Afghanistan,AFG,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5800,Somalia,SOM,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
1043,Cambodia,KHM,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5488,Saudi Arabia,SAU,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5419,San Marino,SMR,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2724,IBRD only,IBD,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4964,Palau,PLW,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4894,Oman,OMN,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4684,New Zealand,NZL,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
1534,Cuba,CUB,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"


In [345]:
excel = pd.read_excel("wiid-data.xlsx")

In [346]:
excel.columns = excel.columns.str.strip()

excel["ISO"] = excel["ISO"].astype(str).str.strip()
excel["YEAR"] = pd.to_numeric(excel["YEAR"], errors="coerce")
excel["GINI INDEX"] = pd.to_numeric(
    excel["GINI INDEX"], errors="coerce"
)

excel = excel.dropna(subset=["ISO", "YEAR", "GINI INDEX"])

In [347]:
gini_data = {}

for _, row in excel.iterrows():

    country = row["ISO"]
    year = int(row["YEAR"])
    value = row["GINI INDEX"]

    if country not in gini_data:
        gini_data[country] = {}

    gini_data[country][year] = value

In [348]:
gini_data = {}

for _, row in excel.iterrows():

    country = row["ISO"]
    year = int(row["YEAR"])
    value = row["GINI INDEX"]

    if country not in gini_data:
        gini_data[country] = {}

    gini_data[country][year] = value

In [349]:
before = gini[years].isna().sum().sum()

for country, values in gini_data.items():

    mask = gini["Country Code"] == country

    for year, value in values.items():

        year = str(year)

        if year in years:
            gini.loc[
                mask & gini[year].isna(), year
            ] = value

after = gini[years].isna().sum().sum()

print("Missing before:", before)
print("Missing after:", after)
print("Values filled:", before - after)

Missing before: 3293
Missing after: 325
Values filled: 2968


In [350]:
gini["Missing Count"] = gini[years].isna().sum(axis=1)

gini["Missing Years"] = gini[years].apply(
    lambda row: ", ".join(
        year for year in years if pd.isna(row[year])
    ),
    axis=1
)

display(
    gini[
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
2823,IDA only,IDX,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3389,"Korea, Dem. People's Rep.",PRK,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2746,IDA & IBRD total,IBT,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2778,IDA blend,IDB,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2724,IBRD only,IBD,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2847,IDA total,IDA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5874,South Asia (IDA & IBRD),TSA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4264,"Middle East, North Africa, Afghanistan & Pakistan (IDA & IBRD)",TMN,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
6106,Sub-Saharan Africa (IDA & IBRD),TSS,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
6138,Sub-Saharan Africa (excluding high income),SSA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"


In [376]:
region = pd.read_excel("wiid-data-region.xlsx")

In [375]:
region_codes = {
    "East Asia and the Pacific": "TEA",
    "Europe and Central Asia": "TEC",
    "Latin America and the Caribbean": "TLA",
    "Middle East, North Africa, Afghanistan and Pakistan": "TMN",
    "North America": "NAC",
    "South Asia": "TSA",
    "Sub-Saharan Africa": "TSS"
}

In [377]:
region["Country Code"] = region["COUNTRY"].map(region_codes)

display(
    region[["COUNTRY", "Country Code"]].drop_duplicates()
)

,COUNTRY,Country Code
0,East Asia and the Pacific,TEA
26,Europe and Central Asia,TEC
52,Latin America and the Caribbean,TLA
78,"Middle East, North Africa, Afghanistan and Pakistan",TMN
104,North America,NAC
130,South Asia,TSA
156,Sub-Saharan Africa,TSS


In [378]:
region_data = {}

for _, row in region.iterrows():

    country = row["Country Code"]
    year = row["YEAR"]
    value = row["GINI INDEX"]

    if pd.notna(country) and pd.notna(year) and pd.notna(value):

        if country not in region_data:
            region_data[country] = {}

        region_data[country][int(year)] = float(value)

In [379]:
before = gini[years].isna().sum().sum()

for country, values in region_data.items():

    mask = gini["Country Code"] == country

    for year, value in values.items():

        year = str(year)

        if year in years:
            gini.loc[
                mask & gini[year].isna(), year
            ] = value

after = gini[years].isna().sum().sum()

print("Values filled:", before - after)
print("Missing remaining:", after)

Values filled: 150
Missing remaining: 175


In [380]:
gini["Missing Count"] = gini[years].isna().sum(axis=1)

display(
    gini[
        gini["Country Code"].isin(region_codes.values())
    ][["Country Name", "Country Code", "Missing Count"]]
)

,Country Name,Country Code,Missing Count
1779,East Asia & Pacific (IDA & IBRD),TEA,0
2094,Europe & Central Asia (IDA & IBRD),TEC,0
3560,Latin America & Caribbean (IDA & IBRD),TLA,0
4264,"Middle East, North Africa, Afghanistan & Pakistan (IDA & IBRD)",TMN,0
5874,South Asia (IDA & IBRD),TSA,0
6106,Sub-Saharan Africa (IDA & IBRD),TSS,0


In [381]:
gini["Missing Count"] = gini[years].isna().sum(axis=1)

gini["Missing Years"] = gini[years].apply(
    lambda row: ", ".join(
        year for year in years if pd.isna(row[year])
    ),
    axis=1
)

display(
    gini[
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
2778,IDA blend,IDB,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3389,"Korea, Dem. People's Rep.",PRK,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2746,IDA & IBRD total,IBT,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
6138,Sub-Saharan Africa (excluding high income),SSA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2823,IDA only,IDX,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2847,IDA total,IDA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2724,IBRD only,IBD,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5104,Peru,PER,0,
5069,Paraguay,PRY,0,
5278,Romania,ROU,0,


In [382]:
gini = gini[
    gini[years].isna().sum(axis=1) < 25
].copy()

print("Gini rows remaining:", len(gini))

Gini rows remaining: 198


In [383]:
drop_rows = df[
    (df["Series Code"] == "SI.POV.GINI") &
    (~df.index.isin(gini.index))
].index

df = df.drop(drop_rows)

df.loc[gini.index, years] = gini[years]

In [384]:
print("Total rows:", len(df))

print("Total missing values:",
      df[years].isna().sum().sum())

print("Missing values by year:")
print(df[years].isna().sum())

Total rows: 6724
Total missing values: 5590
Missing values by year:
2000    368
2001    382
2002    392
2003    398
2004    394
2005    201
2006    195
2007    194
2008    190
2009    191
2010    176
2011    170
2012    178
2013    189
2014    177
2015    175
2016    178
2017    180
2018    173
2019    166
2020    180
2021    178
2022    178
2023    190
2024    197
dtype: int64


In [385]:
print("DataFrame shape:", df.shape)

print("Total missing values:", df.isna().sum().sum())

print("Missing values by column:")
print(df.isna().sum())

DataFrame shape: (6724, 29)
Total missing values: 5590
Missing values by column:
Country Name      0
Country Code      0
Series Name       0
Series Code       0
2000            368
2001            382
2002            392
2003            398
2004            394
2005            201
2006            195
2007            194
2008            190
2009            191
2010            176
2011            170
2012            178
2013            189
2014            177
2015            175
2016            178
2017            180
2018            173
2019            166
2020            180
2021            178
2022            178
2023            190
2024            197
dtype: int64


In [386]:
missing = df[df.isna().any(axis=1)]

display(missing)

,Country Name,Country Code,Series Name,Series Code,2000,2001,2002,2003,2004,2005,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
7,Afghanistan,AFG,"Share of youth not in education, employment or training, total (% of youth population) (modeled ILO estimate)",SL.UEM.NEET.ME.ZS,NaN,NaN,NaN,NaN,NaN,42.3100,...,41.8870,42.3790,42.8400,42.7220,42.5600,53.7640,51.8020,56.0630,55.9950,55.8400
20,Afghanistan,AFG,"Literacy rate, adult total (% of people ages 15 and above)",SE.ADT.LITR.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,33.7500,NaN,NaN,NaN,NaN,36.0150,37.2700,NaN,NaN,NaN
49,Albania,ALB,"Share of youth not in education, employment or training, total (% of youth population) (modeled ILO estimate)",SL.UEM.NEET.ME.ZS,NaN,NaN,NaN,NaN,NaN,35.1860,...,29.8160,27.3910,26.2270,26.6140,25.8240,26.9430,24.0840,23.7140,23.1600,23.2580
63,Albania,ALB,"Literacy rate, adult total (% of people ages 15 and above)",SE.ADT.LITR.ZS,NaN,98.7100,NaN,NaN,NaN,NaN,...,NaN,NaN,98.8100,NaN,NaN,NaN,NaN,NaN,97.6800,NaN
84,Algeria,DZA,"Share of youth not in education, employment or training, total (% of youth population) (modeled ILO estimate)",SL.UEM.NEET.ME.ZS,NaN,NaN,NaN,NaN,NaN,29.2900,...,21.2000,20.4350,20.9500,20.3360,20.0840,20.8690,20.3600,19.8900,19.5600,19.6320
91,Algeria,DZA,"Literacy rate, adult total (% of people ages 15 and above)",SE.ADT.LITR.ZS,NaN,NaN,69.8700,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
119,Andorra,AND,"Share of youth not in education, employment or training, total (% of youth population) (modeled ILO estimate)",SL.UEM.NEET.ME.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
125,Andorra,AND,"Literacy rate, adult total (% of people ages 15 and above)",SE.ADT.LITR.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
154,Angola,AGO,"Share of youth not in education, employment or training, total (% of youth population) (modeled ILO estimate)",SL.UEM.NEET.ME.ZS,NaN,NaN,NaN,NaN,NaN,26.2310,...,29.1090,27.5210,26.0300,24.5820,22.9220,23.5910,21.6390,24.7660,24.6800,24.7390
167,Angola,AGO,"Literacy rate, adult total (% of people ages 15 and above)",SE.ADT.LITR.ZS,NaN,67.4100,NaN,NaN,NaN,NaN,...,66.2400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,68.1800,NaN


In [393]:
missing = df.copy()

missing["Missing Count"] = missing[years].isna().sum(axis=1)

missing_indicators = missing.groupby(
    ["Series Name", "Series Code"]
)["Missing Count"].sum().reset_index()

missing_indicators = missing_indicators[
    missing_indicators["Missing Count"] > 0
]

display(
    missing_indicators.sort_values(
        "Missing Count", ascending=False
    )
)

,Series Name,Series Code,Missing Count
32,"Share of youth not in education, employment or training, total (% of youth population) (modeled ILO estimate)",SL.UEM.NEET.ME.ZS,1332


In [ ]:
neet = df[
    df["Series Code"] == "SL.UEM.NEET.ME.ZS"
].copy()

print("Countries found:", len(neet))

Countries found: 205


In [397]:
neet["Missing Count"] = neet[years].isna().sum(axis=1)

neet["Missing Years"] = neet[years].apply(
    lambda row: ", ".join(
        year for year in years if pd.isna(row[year])
    ),
    axis=1
)

display(
    neet[
        ["Country Name", "Country Code", "Missing Count", "Missing Years"]
    ].sort_values("Missing Count", ascending=False)
)

,Country Name,Country Code,Missing Count,Missing Years
3759,Liechtenstein,LIE,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
5580,Seychelles,SYC,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
2430,Grenada,GRD,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4215,"Micronesia, Fed. Sts.",FSM,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4949,Palau,PLW,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
6699,Tuvalu,TUV,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4074,Marshall Islands,MHL,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
1693,Dominica,DMA,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
3340,Kiribati,KIR,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"
4563,Nauru,NRU,25,"2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024"


In [404]:
neet = neet[
    neet[years].isna().sum(axis=1) < 25
].copy()

print("Countries remaining:", len(neet))

Countries remaining: 190


In [405]:


early_years = [str(y) for y in range(2000, 2005)]

for index in neet.index:

    data = neet.loc[index, years].astype(float)

    # Only predict years before the first observed year
    if data[early_years].isna().any():

        observed = data.dropna()

        if len(observed) >= 5:

            first_year = int(observed.index[0])

            if first_year > 2000:

                try:
                    reversed_data = observed.iloc[::-1]

                    model = ARIMA(reversed_data, order=(1, 1, 0))
                    result = model.fit()

                    predictions = result.forecast(
                        steps=first_year - 2000
                    )

                    for year in early_years:

                        if pd.isna(neet.loc[index, year]):

                            step = first_year - int(year) - 1

                            if 0 <= step < len(predictions):

                                prediction = predictions.iloc[step]

                                if pd.notna(prediction) and 0 <= prediction <= 100:
                                    neet.loc[index, year] = prediction

                except Exception as e:
                    print(neet.loc[index, "Country Name"], e)

c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency -1YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it is not monotonic and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency -1YS-JAN will be used.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it is not monotonic and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\chiam\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency informa

In [406]:


imputer = KNNImputer(n_neighbors=5)

neet[years] = imputer.fit_transform(
    neet[years].astype(float)
)

print("Missing after KNN:", neet[years].isna().sum().sum())

Missing after KNN: 0


In [407]:
drop_rows = df[
    (df["Series Code"] == "SL.UEM.NEET.ME.ZS") &
    (~df.index.isin(neet.index))
].index

df = df.drop(drop_rows)

df.loc[neet.index, years] = neet[years]

In [408]:
print("DataFrame shape:", df.shape)

print("Total missing values:", df.isna().sum().sum())

print("Missing values by column:")
print(df.isna().sum())

DataFrame shape: (6569, 29)
Total missing values: 0
Missing values by column:
Country Name    0
Country Code    0
Series Name     0
Series Code     0
2000            0
2001            0
2002            0
2003            0
2004            0
2005            0
2006            0
2007            0
2008            0
2009            0
2010            0
2011            0
2012            0
2013            0
2014            0
2015            0
2016            0
2017            0
2018            0
2019            0
2020            0
2021            0
2022            0
2023            0
2024            0
dtype: int64


In [409]:
df.to_csv("cleaned_final.csv", index=False)

print("DataFrame saved successfully!")
print("Shape:", df.shape)

DataFrame saved successfully!
Shape: (6569, 29)
